## Imports and Setup 

In [2]:
import  pandas as pd
import numpy as np
import json
from pandas import json_normalize
import matplotlib.pyplot as plt
pd.set_option('future.no_silent_downcasting', True)
import matplotlib.ticker as mticker
import ast
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
import re
from datetime import datetime

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

# Plotting style
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({'figure.dpi': 120, 'axes.titlesize': 13, 'axes.labelsize': 11})
PALETTE   = sns.color_palette('muted')
FIG_BG    = '#F8F9FA'
ACCENT    = '#2B6CB0'
plt.rcParams.update({'figure.facecolor': FIG_BG, 'axes.facecolor': FIG_BG,
                     'figure.dpi': 110, 'axes.spines.top': False,
                     'axes.spines.right': False})

# Primary Data Understanding

In [3]:
MASTER_SCHEMA = [
    # -------------------------
    # 1. Identification
    # -------------------------
    "url",
    "contrat",
    "type",
    "prix",
    "titre",
    "description",

    # -------------------------
    # 3. Price & Physical Features
    # -------------------------
    "surface",
    "pieces",
    "etage",

    # -------------------------
    # 4. Location
    # -------------------------
    "code_postal",
    "adresse",
    "ville",
    "gouvernerat",
    "latitude",
    "longitude",

    # -------------------------
    # 5. Temporal Information
    # -------------------------
    "date_publication",
    "date_scraping",

    # -------------------------
    # 6. Property Quality
    # -------------------------
    "standing",
    "annee_constr",

    # -------------------------
    # 7. Nearby Amenities
    # -------------------------
    "ecole",
    "pharmacie",
    "hopital",
    "marche",
    "magasin",
    "restaurant",
    "bus",
    "railway",
    
    
    "caracteristiques",

    # -------------------------
    # 8. Content & Media
    # -------------------------
    "images"
]

## Monbien.tn

In [4]:
data = pd.read_json('datasets/monbien_tn/data.json')
df_monbien = json_normalize(data.to_dict('records'))
df_monbien.info()
df_monbien

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6939 entries, 0 to 6938
Data columns (total 17 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   source                6939 non-null   object
 1   url                   6939 non-null   object
 2   annonce_id            6939 non-null   int64 
 3   transaction           6939 non-null   object
 4   type_bien             6939 non-null   object
 5   titre                 6939 non-null   object
 6   prix                  6939 non-null   object
 7   surface_m2            6939 non-null   object
 8   pieces                6939 non-null   object
 9   chambres              6939 non-null   object
 10  salles_de_bain        6939 non-null   object
 11  localisation          6939 non-null   object
 12  adresse               6939 non-null   object
 13  description           6939 non-null   object
 14  images_csv            6939 non-null   object
 15  caracteristiques_csv  6939 non-null   

,source,url,annonce_id,transaction,type_bien,titre,prix,surface_m2,pieces,chambres,salles_de_bain,localisation,adresse,description,images_csv,caracteristiques_csv,details_csv
0,monbien.tn,https://www.monbien.tn/properties/7599/terrain...,7599,Vente,Terrain,Terrain de 4018 m² à vendre à 200 dt/m² à hamm...,0 DT,4018,,2,,,,Réf: THS063 Nabeul / Hammamet Terrain / à vend...,https://www.monbien.tn/media/cache/resolve/pho...,Terrain de 4018 m² à vendre à 200 dt/m² à hamm...,
1,monbien.tn,https://www.monbien.tn/properties/7592/coquet-...,7592,Vente,Appartement,Coquet studio à vendre à 140 md à hammamet 513...,140 000 DT,,1,2,,,,Réf: AH077 Nabeul / Hammamet Appartement / à v...,https://www.monbien.tn/media/cache/resolve/pho...,Coquet studio à vendre à 140 md à hammamet 513...,
2,monbien.tn,https://www.monbien.tn/properties/7588/villa-t...,7588,Vente,Villa,Villa toute neuve s+3 avec piscine à hammamet ...,650 000 DT,200,3,2,,,,Réf: MHS186 Nabeul / Hammamet Villa / à vendre...,https://www.monbien.tn/media/cache/resolve/pho...,Villa toute neuve s+3 avec piscine à hammamet ...,
3,monbien.tn,https://www.monbien.tn/properties/7596/un-dupl...,7596,Vente,Terrain,Un duplex avec piscine à hammamet nord à vendr...,730 000 DT,220,3,2,,,,Réf: MH112 Nabeul / Hammamet nord Duplex / à v...,https://www.monbien.tn/media/cache/resolve/pho...,Un duplex avec piscine à hammamet nord à vendr...,
4,monbien.tn,https://www.monbien.tn/properties/7594/apparte...,7594,Vente,Appartement,Appartement s+1 avec vue de mer à hammamet à v...,320 000 DT,76,1,2,,,,Réf: AH079 Nabeul / Hammamet centre Appartemen...,https://www.monbien.tn/media/cache/resolve/pho...,Appartement s+1 avec vue de mer à hammamet à v...,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6934,monbien.tn,https://www.monbien.tn/properties/7077/bel-app...,7077,Vente,Appartement,Bel appartement s+1 à vendre à afh mrezga 5135...,250 000 DT,90,1,2,,,,Réf: AM091 Nabeul / Mrezga Appartement / à ven...,https://www.monbien.tn/media/cache/resolve/pho...,Bel appartement s+1 à vendre à afh mrezga 5135...,
6935,monbien.tn,https://www.monbien.tn/properties/4831/villa-a...,4831,Location,Villa,"Villa azur à louer à sidi daoud, la marsa",4 200 DT,,,2,,,,Réf: OF91933 Tunis / La marsa Villa / à louer ...,https://www.monbien.tn/media/cache/resolve/pho...,"Villa azur à louer à sidi daoud, la marsa | Vi...",
6936,monbien.tn,https://www.monbien.tn/properties/4840/appart-...,4840,Location,Appartement,Appart s3 rano à louer à cité el ghazela,1 100 DT,,,2,,,,Réf: OF92209 Ariana / Ghazela Appartement / à ...,https://www.monbien.tn/media/cache/resolve/pho...,Appart s3 rano à louer à cité el ghazela | App...,
6937,monbien.tn,https://www.monbien.tn/properties/4848/duplex-...,4848,Location,Duplex,Duplex harmonia à louer à golden tulip,2 900 DT,,,2,,,,Réf: OF92205 Tunis / Gammarth Duplex / à louer...,https://www.monbien.tn/media/cache/resolve/pho...,Duplex harmonia à louer à golden tulip | Duple...,


In [5]:
#remove empty strings and common null indicators
df_monbien = df_monbien.replace(
    ["", " ", "NA", "N/A", "na", "null", "None", None],
    np.nan
)
null_table = (
    pd.DataFrame({
        "Null Count": df_monbien.isna().sum(),
        "Null Percentage (%)": df_monbien.isna().mean() * 100
    })
    .round(2)
    .sort_values(by="Null Percentage (%)", ascending=False)
)

print(null_table)


                      Null Count  Null Percentage (%)
details_csv                 6939               100.00
adresse                     6939               100.00
salles_de_bain              6916                99.67
localisation                6752                97.31
pieces                      3360                48.42
surface_m2                  1992                28.71
type_bien                    259                 3.73
chambres                     250                 3.60
source                         0                 0.00
titre                          0                 0.00
prix                           0                 0.00
annonce_id                     0                 0.00
transaction                    0                 0.00
url                            0                 0.00
description                    0                 0.00
images_csv                     0                 0.00
caracteristiques_csv           0                 0.00


### Data Prep

In [6]:
df_monbien_std = df_monbien.drop(columns=[col for col in df_monbien.columns if df_monbien[col].isna().all()])
df_monbien_std.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6939 entries, 0 to 6938
Data columns (total 15 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   source                6939 non-null   object
 1   url                   6939 non-null   object
 2   annonce_id            6939 non-null   int64 
 3   transaction           6939 non-null   object
 4   type_bien             6680 non-null   object
 5   titre                 6939 non-null   object
 6   prix                  6939 non-null   object
 7   surface_m2            4947 non-null   object
 8   pieces                3579 non-null   object
 9   chambres              6689 non-null   object
 10  salles_de_bain        23 non-null     object
 11  localisation          187 non-null    object
 12  description           6939 non-null   object
 13  images_csv            6939 non-null   object
 14  caracteristiques_csv  6939 non-null   object
dtypes: int64(1), object(14)
memory usage: 

In [7]:
df_monbien_std['url'] = df_monbien_std['url'].str.replace(r'-\d{1,3}$', '', regex=True)
df_monbien_std

,source,url,annonce_id,transaction,type_bien,titre,prix,surface_m2,pieces,chambres,salles_de_bain,localisation,description,images_csv,caracteristiques_csv
0,monbien.tn,https://www.monbien.tn/properties/7599/terrain...,7599,Vente,Terrain,Terrain de 4018 m² à vendre à 200 dt/m² à hamm...,0 DT,4018,NaN,2,NaN,NaN,Réf: THS063 Nabeul / Hammamet Terrain / à vend...,https://www.monbien.tn/media/cache/resolve/pho...,Terrain de 4018 m² à vendre à 200 dt/m² à hamm...
1,monbien.tn,https://www.monbien.tn/properties/7592/coquet-...,7592,Vente,Appartement,Coquet studio à vendre à 140 md à hammamet 513...,140 000 DT,NaN,1,2,NaN,NaN,Réf: AH077 Nabeul / Hammamet Appartement / à v...,https://www.monbien.tn/media/cache/resolve/pho...,Coquet studio à vendre à 140 md à hammamet 513...
2,monbien.tn,https://www.monbien.tn/properties/7588/villa-t...,7588,Vente,Villa,Villa toute neuve s+3 avec piscine à hammamet ...,650 000 DT,200,3,2,NaN,NaN,Réf: MHS186 Nabeul / Hammamet Villa / à vendre...,https://www.monbien.tn/media/cache/resolve/pho...,Villa toute neuve s+3 avec piscine à hammamet ...
3,monbien.tn,https://www.monbien.tn/properties/7596/un-dupl...,7596,Vente,Terrain,Un duplex avec piscine à hammamet nord à vendr...,730 000 DT,220,3,2,NaN,NaN,Réf: MH112 Nabeul / Hammamet nord Duplex / à v...,https://www.monbien.tn/media/cache/resolve/pho...,Un duplex avec piscine à hammamet nord à vendr...
4,monbien.tn,https://www.monbien.tn/properties/7594/apparte...,7594,Vente,Appartement,Appartement s+1 avec vue de mer à hammamet à v...,320 000 DT,76,1,2,NaN,NaN,Réf: AH079 Nabeul / Hammamet centre Appartemen...,https://www.monbien.tn/media/cache/resolve/pho...,Appartement s+1 avec vue de mer à hammamet à v...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6934,monbien.tn,https://www.monbien.tn/properties/7077/bel-app...,7077,Vente,Appartement,Bel appartement s+1 à vendre à afh mrezga 5135...,250 000 DT,90,1,2,NaN,NaN,Réf: AM091 Nabeul / Mrezga Appartement / à ven...,https://www.monbien.tn/media/cache/resolve/pho...,Bel appartement s+1 à vendre à afh mrezga 5135...
6935,monbien.tn,https://www.monbien.tn/properties/4831/villa-a...,4831,Location,Villa,"Villa azur à louer à sidi daoud, la marsa",4 200 DT,NaN,NaN,2,NaN,NaN,Réf: OF91933 Tunis / La marsa Villa / à louer ...,https://www.monbien.tn/media/cache/resolve/pho...,"Villa azur à louer à sidi daoud, la marsa | Vi..."
6936,monbien.tn,https://www.monbien.tn/properties/4840/appart-...,4840,Location,Appartement,Appart s3 rano à louer à cité el ghazela,1 100 DT,NaN,NaN,2,NaN,NaN,Réf: OF92209 Ariana / Ghazela Appartement / à ...,https://www.monbien.tn/media/cache/resolve/pho...,Appart s3 rano à louer à cité el ghazela | App...
6937,monbien.tn,https://www.monbien.tn/properties/4848/duplex-...,4848,Location,Duplex,Duplex harmonia à louer à golden tulip,2 900 DT,NaN,NaN,2,NaN,NaN,Réf: OF92205 Tunis / Gammarth Duplex / à louer...,https://www.monbien.tn/media/cache/resolve/pho...,Duplex harmonia à louer à golden tulip | Duple...


In [8]:
# 2. Extract the Property ID as a numeric value for comparison
# This looks for the digits between '/properties/' and the next '/'
df_monbien_std['prop_id'] = df_monbien_std['url'].str.extract(r'/properties/(\d+)/').astype(int)

# 3. Sort by Property ID (Descending) so the biggest is at the top
df_monbien_std = df_monbien_std.sort_values('prop_id', ascending=False)

# 4. Drop duplicates based on the URL, keeping the first (which is now the highest ID)
df_monbien_std = df_monbien_std.drop_duplicates(subset='titre', keep='first')

# Optional: Remove the temporary 'prop_id' column if you don't need it anymore
# df_monbien_std = df_monbien_std.drop(columns=['prop_id'])
df_monbien_std


,source,url,annonce_id,transaction,type_bien,titre,prix,surface_m2,pieces,chambres,salles_de_bain,localisation,description,images_csv,caracteristiques_csv,prop_id
10,monbien.tn,https://www.monbien.tn/properties/7608/terrain...,7608,Vente,Terrain,Terrain de 400 m² avec vue de mer à hammamet s...,60 000 DT,400,NaN,2,NaN,NaN,Réf: THS126 Nabeul / Hammamet Terrain / à vend...,https://www.monbien.tn/media/cache/resolve/pho...,Terrain de 400 m² avec vue de mer à hammamet s...,7608
6913,monbien.tn,https://www.monbien.tn/properties/7607/maison-...,7607,Vente,NaN,Maison s+2 avec garage et jardin à hammamet su...,180 000 DT,NaN,2,2,NaN,NaN,Réf: MHS149 Nabeul / Hammamet Maison / à vendr...,https://www.monbien.tn/media/cache/resolve/pho...,Maison s+2 avec garage et jardin à hammamet su...,7607
8,monbien.tn,https://www.monbien.tn/properties/7606/villa-s...,7606,Vente,Villa,Villa s+5 toute neuve avec piscine à vendre à ...,790 000 DT,500,5,3,NaN,NaN,Réf: MHS139 Nabeul / Hammamet Villa / à vendre...,https://www.monbien.tn/media/cache/resolve/pho...,Villa s+5 toute neuve avec piscine à vendre à ...,7606
9,monbien.tn,https://www.monbien.tn/properties/7605/terrain...,7605,Vente,Terrain,Terrain de 300 m² à hammamet sud à vendre 5135...,60 000 DT,300,NaN,2,NaN,NaN,Réf: THS157 Nabeul / Hammamet Terrain / à vend...,https://www.monbien.tn/media/cache/resolve/pho...,Terrain de 300 m² à hammamet sud à vendre 5135...,7605
13,monbien.tn,https://www.monbien.tn/properties/7604/un-dupl...,7604,Vente,Terrain,Un duplex s+5 avec piscine à hammamet sud à ve...,650 000 DT,250,5,2,NaN,NaN,Réf: MHS132 Nabeul / Hammamet Duplex / à vendr...,https://www.monbien.tn/media/cache/resolve/pho...,Un duplex s+5 avec piscine à hammamet sud à ve...,7604
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3692,monbien.tn,https://www.monbien.tn/properties/5/terrain-80...,5,Vente,Terrain,"Terrain 8073 m² proche du golf citrus, vue mer...",420 000 DT,8073,NaN,2,NaN,NaN,Nabeul / Hammamet Terrain / à vendre il y a 1 ...,https://www.monbien.tn/media/cache/resolve/pho...,"Terrain 8073 m² proche du golf citrus, vue mer...",5
3695,monbien.tn,https://www.monbien.tn/properties/4/excellent-...,4,Vente,Appartement,Excellent investissement à mrezga hammamet : v...,2 600 000 DT,400,NaN,2,NaN,NaN,Nabeul / Hammamet Autre / à vendre il y a 1 an...,https://www.monbien.tn/media/cache/resolve/pho...,Excellent investissement à mrezga hammamet : v...,4
4141,monbien.tn,https://www.monbien.tn/properties/3/excellent-...,3,Vente,Terrain,"Excellent terrain viabilisé à nabeul ville, ci...",363 000 DT,278,NaN,2,NaN,NaN,Nabeul / Nabeul Terrain / à vendre il y a 1 an...,https://www.monbien.tn/media/cache/resolve/pho...,"Excellent terrain viabilisé à nabeul ville, ci...",3
3936,monbien.tn,https://www.monbien.tn/properties/2/magnifique...,2,Vente,Villa,"Magnifique villa toute neuve à hammamet sud, s...",530 000 DT,340,5,2,NaN,NaN,Nabeul / Hammamet Villa / à vendre il y a 1 an...,https://www.monbien.tn/media/cache/resolve/pho...,"Magnifique villa toute neuve à hammamet sud, s...",2


In [9]:
def normalize_schema(df, mapping):
    # rename columns to master names
    df = df.rename(columns=mapping)

    # add missing columns
    for col in MASTER_SCHEMA:
        if col not in df.columns:
            df[col] = None

    # keep only master schema order
    df = df[MASTER_SCHEMA]

    return df

In [10]:
mapping_monbien = {
    "url": "url",
    "transaction": "contrat",
    "description": "description",
    "titre": "titre",
    "pieces": "pieces",
    "prix": "prix",
    "type_bien": "type",
    "surface_m2": "surface",
    "localisation": "ville",
    "caracteristiques_csv": "caracteristiques",
    "images_csv": "images"
}
df_monbien_std = normalize_schema(df_monbien_std, mapping_monbien)
df_monbien_std.info()
df_monbien_std

<class 'pandas.core.frame.DataFrame'>
Index: 2969 entries, 10 to 3935
Data columns (total 29 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   url               2969 non-null   object
 1   contrat           2969 non-null   object
 2   type              2861 non-null   object
 3   prix              2969 non-null   object
 4   titre             2969 non-null   object
 5   description       2969 non-null   object
 6   surface           1498 non-null   object
 7   pieces            1361 non-null   object
 8   etage             0 non-null      object
 9   code_postal       0 non-null      object
 10  adresse           0 non-null      object
 11  ville             185 non-null    object
 12  gouvernerat       0 non-null      object
 13  latitude          0 non-null      object
 14  longitude         0 non-null      object
 15  date_publication  0 non-null      object
 16  date_scraping     0 non-null      object
 17  standing          

,url,contrat,type,prix,titre,description,surface,pieces,etage,code_postal,adresse,ville,gouvernerat,latitude,longitude,date_publication,date_scraping,standing,annee_constr,ecole,pharmacie,hopital,marche,magasin,restaurant,bus,railway,caracteristiques,images
10,https://www.monbien.tn/properties/7608/terrain...,Vente,Terrain,60 000 DT,Terrain de 400 m² avec vue de mer à hammamet s...,Réf: THS126 Nabeul / Hammamet Terrain / à vend...,400,NaN,None,None,None,NaN,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,Terrain de 400 m² avec vue de mer à hammamet s...,https://www.monbien.tn/media/cache/resolve/pho...
6913,https://www.monbien.tn/properties/7607/maison-...,Vente,NaN,180 000 DT,Maison s+2 avec garage et jardin à hammamet su...,Réf: MHS149 Nabeul / Hammamet Maison / à vendr...,NaN,2,None,None,None,NaN,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,Maison s+2 avec garage et jardin à hammamet su...,https://www.monbien.tn/media/cache/resolve/pho...
8,https://www.monbien.tn/properties/7606/villa-s...,Vente,Villa,790 000 DT,Villa s+5 toute neuve avec piscine à vendre à ...,Réf: MHS139 Nabeul / Hammamet Villa / à vendre...,500,5,None,None,None,NaN,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,Villa s+5 toute neuve avec piscine à vendre à ...,https://www.monbien.tn/media/cache/resolve/pho...
9,https://www.monbien.tn/properties/7605/terrain...,Vente,Terrain,60 000 DT,Terrain de 300 m² à hammamet sud à vendre 5135...,Réf: THS157 Nabeul / Hammamet Terrain / à vend...,300,NaN,None,None,None,NaN,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,Terrain de 300 m² à hammamet sud à vendre 5135...,https://www.monbien.tn/media/cache/resolve/pho...
13,https://www.monbien.tn/properties/7604/un-dupl...,Vente,Terrain,650 000 DT,Un duplex s+5 avec piscine à hammamet sud à ve...,Réf: MHS132 Nabeul / Hammamet Duplex / à vendr...,250,5,None,None,None,NaN,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,Un duplex s+5 avec piscine à hammamet sud à ve...,https://www.monbien.tn/media/cache/resolve/pho...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3692,https://www.monbien.tn/properties/5/terrain-80...,Vente,Terrain,420 000 DT,"Terrain 8073 m² proche du golf citrus, vue mer...",Nabeul / Hammamet Terrain / à vendre il y a 1 ...,8073,NaN,None,None,None,NaN,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,"Terrain 8073 m² proche du golf citrus, vue mer...",https://www.monbien.tn/media/cache/resolve/pho...
3695,https://www.monbien.tn/properties/4/excellent-...,Vente,Appartement,2 600 000 DT,Excellent investissement à mrezga hammamet : v...,Nabeul / Hammamet Autre / à vendre il y a 1 an...,400,NaN,None,None,None,NaN,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,Excellent investissement à mrezga hammamet : v...,https://www.monbien.tn/media/cache/resolve/pho...
4141,https://www.monbien.tn/properties/3/excellent-...,Vente,Terrain,363 000 DT,"Excellent terrain viabilisé à nabeul ville, ci...",Nabeul / Nabeul Terrain / à vendre il y a 1 an...,278,NaN,None,None,None,NaN,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,"Excellent terrain viabilisé à nabeul ville, ci...",https://www.monbien.tn/media/cache/resolve/pho...
3936,https://www.monbien.tn/properties/2/magnifique...,Vente,Villa,530 000 DT,"Magnifique villa toute neuve à hammamet sud, s...",Nabeul / Hammamet Villa / à vendre il y a 1 an...,340,5,None,None,None,NaN,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,"Magnifique villa toute neuve à hammamet sud, s...",https://www.monbien.tn/media/cache/resolve/pho...


In [11]:
df_monbien_std["description"] = np.where(
    df_monbien_std["caracteristiques"].notna(),
    df_monbien_std["description"].fillna("") +
    "\nCaracteristiques: " +
    df_monbien_std["caracteristiques"],
    df_monbien_std["description"]
)

df_monbien_std["caracteristiques"] = np.nan
df_monbien_std

,url,contrat,type,prix,titre,description,surface,pieces,etage,code_postal,adresse,ville,gouvernerat,latitude,longitude,date_publication,date_scraping,standing,annee_constr,ecole,pharmacie,hopital,marche,magasin,restaurant,bus,railway,caracteristiques,images
10,https://www.monbien.tn/properties/7608/terrain...,Vente,Terrain,60 000 DT,Terrain de 400 m² avec vue de mer à hammamet s...,Réf: THS126 Nabeul / Hammamet Terrain / à vend...,400,NaN,None,None,None,NaN,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,NaN,https://www.monbien.tn/media/cache/resolve/pho...
6913,https://www.monbien.tn/properties/7607/maison-...,Vente,NaN,180 000 DT,Maison s+2 avec garage et jardin à hammamet su...,Réf: MHS149 Nabeul / Hammamet Maison / à vendr...,NaN,2,None,None,None,NaN,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,NaN,https://www.monbien.tn/media/cache/resolve/pho...
8,https://www.monbien.tn/properties/7606/villa-s...,Vente,Villa,790 000 DT,Villa s+5 toute neuve avec piscine à vendre à ...,Réf: MHS139 Nabeul / Hammamet Villa / à vendre...,500,5,None,None,None,NaN,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,NaN,https://www.monbien.tn/media/cache/resolve/pho...
9,https://www.monbien.tn/properties/7605/terrain...,Vente,Terrain,60 000 DT,Terrain de 300 m² à hammamet sud à vendre 5135...,Réf: THS157 Nabeul / Hammamet Terrain / à vend...,300,NaN,None,None,None,NaN,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,NaN,https://www.monbien.tn/media/cache/resolve/pho...
13,https://www.monbien.tn/properties/7604/un-dupl...,Vente,Terrain,650 000 DT,Un duplex s+5 avec piscine à hammamet sud à ve...,Réf: MHS132 Nabeul / Hammamet Duplex / à vendr...,250,5,None,None,None,NaN,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,NaN,https://www.monbien.tn/media/cache/resolve/pho...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3692,https://www.monbien.tn/properties/5/terrain-80...,Vente,Terrain,420 000 DT,"Terrain 8073 m² proche du golf citrus, vue mer...",Nabeul / Hammamet Terrain / à vendre il y a 1 ...,8073,NaN,None,None,None,NaN,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,NaN,https://www.monbien.tn/media/cache/resolve/pho...
3695,https://www.monbien.tn/properties/4/excellent-...,Vente,Appartement,2 600 000 DT,Excellent investissement à mrezga hammamet : v...,Nabeul / Hammamet Autre / à vendre il y a 1 an...,400,NaN,None,None,None,NaN,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,NaN,https://www.monbien.tn/media/cache/resolve/pho...
4141,https://www.monbien.tn/properties/3/excellent-...,Vente,Terrain,363 000 DT,"Excellent terrain viabilisé à nabeul ville, ci...",Nabeul / Nabeul Terrain / à vendre il y a 1 an...,278,NaN,None,None,None,NaN,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,NaN,https://www.monbien.tn/media/cache/resolve/pho...
3936,https://www.monbien.tn/properties/2/magnifique...,Vente,Villa,530 000 DT,"Magnifique villa toute neuve à hammamet sud, s...",Nabeul / Hammamet Villa / à vendre il y a 1 an...,340,5,None,None,None,NaN,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,NaN,https://www.monbien.tn/media/cache/resolve/pho...


#### Data type conversion

In [12]:
df_monbien_std["prix"] = (
    df_monbien_std["prix"]
    .astype(str)
    .str.replace(r"[^\d.]", "", regex=True)  # keep only numbers
)

df_monbien_std["prix"] = pd.to_numeric(df_monbien_std["prix"], errors="coerce")

df_monbien_std["surface"] = pd.to_numeric(df_monbien_std["surface"], errors="coerce")

df_monbien_std["etage"] = pd.to_numeric(df_monbien_std["etage"], errors="coerce")
df_monbien_std["annee_constr"] = pd.to_numeric(df_monbien_std["annee_constr"], errors="coerce")

df_monbien_std["pieces"] = pd.to_numeric(df_monbien_std["pieces"], errors="coerce")
df_monbien_std["pieces"] += 1

df_monbien_std["date_publication"] = pd.to_datetime(df_monbien_std["date_publication"], errors="coerce")

df_monbien_std

,url,contrat,type,prix,titre,description,surface,pieces,etage,code_postal,adresse,ville,gouvernerat,latitude,longitude,date_publication,date_scraping,standing,annee_constr,ecole,pharmacie,hopital,marche,magasin,restaurant,bus,railway,caracteristiques,images
10,https://www.monbien.tn/properties/7608/terrain...,Vente,Terrain,60000,Terrain de 400 m² avec vue de mer à hammamet s...,Réf: THS126 Nabeul / Hammamet Terrain / à vend...,400.00,NaN,NaN,None,None,NaN,None,None,None,NaT,None,None,NaN,None,None,None,None,None,None,None,None,NaN,https://www.monbien.tn/media/cache/resolve/pho...
6913,https://www.monbien.tn/properties/7607/maison-...,Vente,NaN,180000,Maison s+2 avec garage et jardin à hammamet su...,Réf: MHS149 Nabeul / Hammamet Maison / à vendr...,NaN,3.00,NaN,None,None,NaN,None,None,None,NaT,None,None,NaN,None,None,None,None,None,None,None,None,NaN,https://www.monbien.tn/media/cache/resolve/pho...
8,https://www.monbien.tn/properties/7606/villa-s...,Vente,Villa,790000,Villa s+5 toute neuve avec piscine à vendre à ...,Réf: MHS139 Nabeul / Hammamet Villa / à vendre...,500.00,6.00,NaN,None,None,NaN,None,None,None,NaT,None,None,NaN,None,None,None,None,None,None,None,None,NaN,https://www.monbien.tn/media/cache/resolve/pho...
9,https://www.monbien.tn/properties/7605/terrain...,Vente,Terrain,60000,Terrain de 300 m² à hammamet sud à vendre 5135...,Réf: THS157 Nabeul / Hammamet Terrain / à vend...,300.00,NaN,NaN,None,None,NaN,None,None,None,NaT,None,None,NaN,None,None,None,None,None,None,None,None,NaN,https://www.monbien.tn/media/cache/resolve/pho...
13,https://www.monbien.tn/properties/7604/un-dupl...,Vente,Terrain,650000,Un duplex s+5 avec piscine à hammamet sud à ve...,Réf: MHS132 Nabeul / Hammamet Duplex / à vendr...,250.00,6.00,NaN,None,None,NaN,None,None,None,NaT,None,None,NaN,None,None,None,None,None,None,None,None,NaN,https://www.monbien.tn/media/cache/resolve/pho...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3692,https://www.monbien.tn/properties/5/terrain-80...,Vente,Terrain,420000,"Terrain 8073 m² proche du golf citrus, vue mer...",Nabeul / Hammamet Terrain / à vendre il y a 1 ...,8073.00,NaN,NaN,None,None,NaN,None,None,None,NaT,None,None,NaN,None,None,None,None,None,None,None,None,NaN,https://www.monbien.tn/media/cache/resolve/pho...
3695,https://www.monbien.tn/properties/4/excellent-...,Vente,Appartement,2600000,Excellent investissement à mrezga hammamet : v...,Nabeul / Hammamet Autre / à vendre il y a 1 an...,400.00,NaN,NaN,None,None,NaN,None,None,None,NaT,None,None,NaN,None,None,None,None,None,None,None,None,NaN,https://www.monbien.tn/media/cache/resolve/pho...
4141,https://www.monbien.tn/properties/3/excellent-...,Vente,Terrain,363000,"Excellent terrain viabilisé à nabeul ville, ci...",Nabeul / Nabeul Terrain / à vendre il y a 1 an...,278.00,NaN,NaN,None,None,NaN,None,None,None,NaT,None,None,NaN,None,None,None,None,None,None,None,None,NaN,https://www.monbien.tn/media/cache/resolve/pho...
3936,https://www.monbien.tn/properties/2/magnifique...,Vente,Villa,530000,"Magnifique villa toute neuve à hammamet sud, s...",Nabeul / Hammamet Villa / à vendre il y a 1 an...,340.00,6.00,NaN,None,None,NaN,None,None,None,NaT,None,None,NaN,None,None,None,None,None,None,None,None,NaN,https://www.monbien.tn/media/cache/resolve/pho...


## Immobilier.tn

In [13]:
data = pd.read_json('datasets/immobilier_com_tn/data.json')
df_immobilier_tn = json_normalize(data.to_dict('records'))
df_immobilier_tn.info()
df_immobilier_tn

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 896 entries, 0 to 895
Data columns (total 17 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   source                896 non-null    object
 1   url                   896 non-null    object
 2   annonce_id            896 non-null    int64 
 3   transaction           896 non-null    object
 4   type_bien             896 non-null    object
 5   titre                 896 non-null    object
 6   prix                  896 non-null    object
 7   surface_m2            896 non-null    object
 8   pieces                896 non-null    object
 9   chambres              896 non-null    object
 10  salles_de_bain        896 non-null    object
 11  localisation          896 non-null    object
 12  adresse               896 non-null    object
 13  description           896 non-null    object
 14  images_csv            896 non-null    object
 15  caracteristiques_csv  896 non-null    ob

,source,url,annonce_id,transaction,type_bien,titre,prix,surface_m2,pieces,chambres,salles_de_bain,localisation,adresse,description,images_csv,caracteristiques_csv,details_csv
0,immobilier.com.tn,https://www.immobilier.com.tn/annonce/33625/lo...,33625,Location,Bureau,Location : Local bureautique à louer (Marsa Ma...,2 300 DT,148,,,,Tunis,rue }; Bureau 148 m² 4 Nous vous proposons à l...,Local bureautique à louer (Marsa Mall) Sidi Da...,,Bureau | 148 m² | Étage 1er étage | Climatisat...,
1,immobilier.com.tn,https://www.immobilier.com.tn/annonce/33645/a-...,33645,Location,Appartement,Location : A louer Etage de villa à Jardin d’E...,1 850 DT,180,3,,,Ariana,rue }; Appartement 180 m² 3 À louer – Étage de...,A louer Etage de villa à Jardin d’El menzah 2 ...,,Appartement | 180 m² | Chauffage Central | Éta...,
2,immobilier.com.tn,https://www.immobilier.com.tn/annonce/33502/al...,33502,Location,Bureau,Location : AL Bureau 346m² au Lac1 - immobilie...,10 700 DT,346,,,,Tunis - Lac1,"Bloc B, 4ème étage, Immeuble Constance, Rue du...","Bloc B, 4ème étage, Immeuble Constance, Rue du...",,Bureau | 346 m²,
3,immobilier.com.tn,https://www.immobilier.com.tn/annonce/33651/bu...,33651,Location,Appartement,Location : bureau haut standing - Pacha centre...,,90,,,,Tunis,rue }; Appartement 90 m² 3 A louer un bureau h...,bureau haut standing - Pacha centre Pacha cent...,,Appartement | 90 m² | Parking | Alarme | Ascen...,
4,immobilier.com.tn,https://www.immobilier.com.tn/annonce/33619/s1...,33619,Location,Appartement,Location : S+1 à louer - La Marsa (cité el Kha...,1 000 DT,43,1,,,Tunis,rue }; Appartement 43 m² Studio Cet appartemen...,S+1 à louer - La Marsa (cité el Khalil) Cite E...,,Appartement | 43 m² | Studio | Chauffage Gaz |...,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
891,immobilier.com.tn,https://www.immobilier.com.tn/annonce/33637/ap...,33637,Location,Appartement,Location : appartement deux chambres - immobi...,650 DT,82,2,,,Ariana,rue }; Appartement 82 m² 2 Appartement propre ...,"appartement deux chambres Cite Ennour Jaafar, ...",,Appartement | 82 m² | Chauffage | Étage | Park...,
892,immobilier.com.tn,https://www.immobilier.com.tn/annonce/33503/al...,33503,Location,Bureau,Location : AL Bureau au lac3 - immobilier.com.tn,8 200 DT,492,,,,Tunis,"Bloc B, 4ème étage, Immeuble Constance, Rue du...","Bloc B, 4ème étage, Immeuble Constance, Rue du...",,Bureau | 492 m²,
893,immobilier.com.tn,https://www.immobilier.com.tn/annonce/33624/au...,33624,Location,Appartement,Location : Au coeur de la Marsa - immobilier....,4 200 DT,80,2,,,Tunis,"rue }; Appartement 80 m² 2 À louer à la Marsa,...","Au coeur de la Marsa Marsa Safsaf, La Marsa, T...",,Appartement | 80 m² | Meublé | Chauffage | Rez...,
894,immobilier.com.tn,https://www.immobilier.com.tn/annonce/33648/a-...,33648,Location,Villa,Location : A louer duplex résidence Riadh Borj...,,124,,,,Ariana,rue }; Villa 124 m² 4 À louer – Duplex à Résid...,A louer duplex résidence Riadh Borj touil Raou...,,Villa | 124 m²,


In [14]:
df_immobilier_tn = df_immobilier_tn.replace(
    ["", " ", "NA", "N/A", "na", "null", "None", None],
    np.nan
)
null_table = (
    pd.DataFrame({
        "Null Count": df_immobilier_tn.isna().sum(),
        "Null Percentage (%)": df_immobilier_tn.isna().mean() * 100
    })
    .round(2)
    .sort_values(by="Null Percentage (%)", ascending=False)
)
print(null_table)

                      Null Count  Null Percentage (%)
salles_de_bain               896               100.00
chambres                     896               100.00
details_csv                  896               100.00
localisation                 458                51.12
pieces                       342                38.17
surface_m2                   179                19.98
adresse                       92                10.27
images_csv                    31                 3.46
prix                          15                 1.67
transaction                    2                 0.22
type_bien                      1                 0.11
source                         0                 0.00
annonce_id                     0                 0.00
titre                          0                 0.00
url                            0                 0.00
description                    0                 0.00
caracteristiques_csv           0                 0.00


### Data Prep

In [15]:
df_immobilier_tn_std = df_immobilier_tn.drop(columns=[col for col in df_immobilier_tn.columns if df_immobilier_tn[col].isna().all()])
df_immobilier_tn_std.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 896 entries, 0 to 895
Data columns (total 14 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   source                896 non-null    object
 1   url                   896 non-null    object
 2   annonce_id            896 non-null    int64 
 3   transaction           894 non-null    object
 4   type_bien             895 non-null    object
 5   titre                 896 non-null    object
 6   prix                  881 non-null    object
 7   surface_m2            717 non-null    object
 8   pieces                554 non-null    object
 9   localisation          438 non-null    object
 10  adresse               804 non-null    object
 11  description           896 non-null    object
 12  images_csv            865 non-null    object
 13  caracteristiques_csv  896 non-null    object
dtypes: int64(1), object(13)
memory usage: 98.1+ KB


In [16]:
def normalize_schema(df, mapping):
    # rename columns to master names
    df = df.rename(columns=mapping)

    # add missing columns
    for col in MASTER_SCHEMA:
        if col not in df.columns:
            df[col] = None

    # keep only master schema order
    df = df[MASTER_SCHEMA]

    return df

In [17]:
mapping_immobilier_tn = {
    "url": "url",
    "transaction": "contrat",
    "description": "description",
    "titre": "titre",
    "pieces": "pieces",
    "prix": "prix",
    "adresse": "adresse",
    "type_bien": "type",
    "surface_m2": "surface",
    "localisation": "ville",
    "caracteristiques_csv": "caracteristiques",
    "images_csv": "images"
}
df_immobilier_tn_std = normalize_schema(df_immobilier_tn_std, mapping_immobilier_tn)
df_immobilier_tn_std.info()
df_immobilier_tn_std

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 896 entries, 0 to 895
Data columns (total 29 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   url               896 non-null    object
 1   contrat           894 non-null    object
 2   type              895 non-null    object
 3   prix              881 non-null    object
 4   titre             896 non-null    object
 5   description       896 non-null    object
 6   surface           717 non-null    object
 7   pieces            554 non-null    object
 8   etage             0 non-null      object
 9   code_postal       0 non-null      object
 10  adresse           804 non-null    object
 11  ville             438 non-null    object
 12  gouvernerat       0 non-null      object
 13  latitude          0 non-null      object
 14  longitude         0 non-null      object
 15  date_publication  0 non-null      object
 16  date_scraping     0 non-null      object
 17  standing        

,url,contrat,type,prix,titre,description,surface,pieces,etage,code_postal,adresse,ville,gouvernerat,latitude,longitude,date_publication,date_scraping,standing,annee_constr,ecole,pharmacie,hopital,marche,magasin,restaurant,bus,railway,caracteristiques,images
0,https://www.immobilier.com.tn/annonce/33625/lo...,Location,Bureau,2 300 DT,Location : Local bureautique à louer (Marsa Ma...,Local bureautique à louer (Marsa Mall) Sidi Da...,148,NaN,None,None,rue }; Bureau 148 m² 4 Nous vous proposons à l...,Tunis,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,Bureau | 148 m² | Étage 1er étage | Climatisat...,NaN
1,https://www.immobilier.com.tn/annonce/33645/a-...,Location,Appartement,1 850 DT,Location : A louer Etage de villa à Jardin d’E...,A louer Etage de villa à Jardin d’El menzah 2 ...,180,3,None,None,rue }; Appartement 180 m² 3 À louer – Étage de...,Ariana,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,Appartement | 180 m² | Chauffage Central | Éta...,NaN
2,https://www.immobilier.com.tn/annonce/33502/al...,Location,Bureau,10 700 DT,Location : AL Bureau 346m² au Lac1 - immobilie...,"Bloc B, 4ème étage, Immeuble Constance, Rue du...",346,NaN,None,None,"Bloc B, 4ème étage, Immeuble Constance, Rue du...",Tunis - Lac1,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,Bureau | 346 m²,NaN
3,https://www.immobilier.com.tn/annonce/33651/bu...,Location,Appartement,NaN,Location : bureau haut standing - Pacha centre...,bureau haut standing - Pacha centre Pacha cent...,90,NaN,None,None,rue }; Appartement 90 m² 3 A louer un bureau h...,Tunis,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,Appartement | 90 m² | Parking | Alarme | Ascen...,NaN
4,https://www.immobilier.com.tn/annonce/33619/s1...,Location,Appartement,1 000 DT,Location : S+1 à louer - La Marsa (cité el Kha...,S+1 à louer - La Marsa (cité el Khalil) Cite E...,43,1,None,None,rue }; Appartement 43 m² Studio Cet appartemen...,Tunis,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,Appartement | 43 m² | Studio | Chauffage Gaz |...,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
891,https://www.immobilier.com.tn/annonce/33637/ap...,Location,Appartement,650 DT,Location : appartement deux chambres - immobi...,"appartement deux chambres Cite Ennour Jaafar, ...",82,2,None,None,rue }; Appartement 82 m² 2 Appartement propre ...,Ariana,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,Appartement | 82 m² | Chauffage | Étage | Park...,NaN
892,https://www.immobilier.com.tn/annonce/33503/al...,Location,Bureau,8 200 DT,Location : AL Bureau au lac3 - immobilier.com.tn,"Bloc B, 4ème étage, Immeuble Constance, Rue du...",492,NaN,None,None,"Bloc B, 4ème étage, Immeuble Constance, Rue du...",Tunis,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,Bureau | 492 m²,NaN
893,https://www.immobilier.com.tn/annonce/33624/au...,Location,Appartement,4 200 DT,Location : Au coeur de la Marsa - immobilier....,"Au coeur de la Marsa Marsa Safsaf, La Marsa, T...",80,2,None,None,"rue }; Appartement 80 m² 2 À louer à la Marsa,...",Tunis,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,Appartement | 80 m² | Meublé | Chauffage | Rez...,NaN
894,https://www.immobilier.com.tn/annonce/33648/a-...,Location,Villa,NaN,Location : A louer duplex résidence Riadh Borj...,A louer duplex résidence Riadh Borj touil Raou...,124,NaN,None,None,rue }; Villa 124 m² 4 À louer – Duplex à Résid...,Ariana,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,Villa | 124 m²,NaN


In [18]:
df_immobilier_tn_std["description"] = np.where(
    df_immobilier_tn_std["caracteristiques"].notna(),
    df_immobilier_tn_std["description"].fillna("") +
    "\nCaracteristiques: " +
    df_immobilier_tn_std["caracteristiques"],
    df_immobilier_tn_std["description"]
)

df_immobilier_tn_std["caracteristiques"] = np.nan
df_immobilier_tn_std

,url,contrat,type,prix,titre,description,surface,pieces,etage,code_postal,adresse,ville,gouvernerat,latitude,longitude,date_publication,date_scraping,standing,annee_constr,ecole,pharmacie,hopital,marche,magasin,restaurant,bus,railway,caracteristiques,images
0,https://www.immobilier.com.tn/annonce/33625/lo...,Location,Bureau,2 300 DT,Location : Local bureautique à louer (Marsa Ma...,Local bureautique à louer (Marsa Mall) Sidi Da...,148,NaN,None,None,rue }; Bureau 148 m² 4 Nous vous proposons à l...,Tunis,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,NaN,NaN
1,https://www.immobilier.com.tn/annonce/33645/a-...,Location,Appartement,1 850 DT,Location : A louer Etage de villa à Jardin d’E...,A louer Etage de villa à Jardin d’El menzah 2 ...,180,3,None,None,rue }; Appartement 180 m² 3 À louer – Étage de...,Ariana,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,NaN,NaN
2,https://www.immobilier.com.tn/annonce/33502/al...,Location,Bureau,10 700 DT,Location : AL Bureau 346m² au Lac1 - immobilie...,"Bloc B, 4ème étage, Immeuble Constance, Rue du...",346,NaN,None,None,"Bloc B, 4ème étage, Immeuble Constance, Rue du...",Tunis - Lac1,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,NaN,NaN
3,https://www.immobilier.com.tn/annonce/33651/bu...,Location,Appartement,NaN,Location : bureau haut standing - Pacha centre...,bureau haut standing - Pacha centre Pacha cent...,90,NaN,None,None,rue }; Appartement 90 m² 3 A louer un bureau h...,Tunis,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,NaN,NaN
4,https://www.immobilier.com.tn/annonce/33619/s1...,Location,Appartement,1 000 DT,Location : S+1 à louer - La Marsa (cité el Kha...,S+1 à louer - La Marsa (cité el Khalil) Cite E...,43,1,None,None,rue }; Appartement 43 m² Studio Cet appartemen...,Tunis,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
891,https://www.immobilier.com.tn/annonce/33637/ap...,Location,Appartement,650 DT,Location : appartement deux chambres - immobi...,"appartement deux chambres Cite Ennour Jaafar, ...",82,2,None,None,rue }; Appartement 82 m² 2 Appartement propre ...,Ariana,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,NaN,NaN
892,https://www.immobilier.com.tn/annonce/33503/al...,Location,Bureau,8 200 DT,Location : AL Bureau au lac3 - immobilier.com.tn,"Bloc B, 4ème étage, Immeuble Constance, Rue du...",492,NaN,None,None,"Bloc B, 4ème étage, Immeuble Constance, Rue du...",Tunis,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,NaN,NaN
893,https://www.immobilier.com.tn/annonce/33624/au...,Location,Appartement,4 200 DT,Location : Au coeur de la Marsa - immobilier....,"Au coeur de la Marsa Marsa Safsaf, La Marsa, T...",80,2,None,None,"rue }; Appartement 80 m² 2 À louer à la Marsa,...",Tunis,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,NaN,NaN
894,https://www.immobilier.com.tn/annonce/33648/a-...,Location,Villa,NaN,Location : A louer duplex résidence Riadh Borj...,A louer duplex résidence Riadh Borj touil Raou...,124,NaN,None,None,rue }; Villa 124 m² 4 À louer – Duplex à Résid...,Ariana,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,NaN,NaN


#### Data type conversion

In [19]:
df_immobilier_tn_std["prix"] = (
    df_immobilier_tn_std["prix"]
    .astype(str)
    .str.replace(r"[^\d.]", "", regex=True)  # keep only numbers
)

df_immobilier_tn_std["prix"] = pd.to_numeric(df_immobilier_tn_std["prix"], errors="coerce")

df_immobilier_tn_std["surface"] = pd.to_numeric(df_immobilier_tn_std["surface"], errors="coerce")

df_immobilier_tn_std["etage"] = pd.to_numeric(df_immobilier_tn_std["etage"], errors="coerce")
df_immobilier_tn_std["annee_constr"] = pd.to_numeric(df_immobilier_tn_std["annee_constr"], errors="coerce")

df_immobilier_tn_std["pieces"] = pd.to_numeric(df_immobilier_tn_std["pieces"], errors="coerce")
df_immobilier_tn_std["pieces"] += 1

df_immobilier_tn_std["date_publication"] = pd.to_datetime(df_immobilier_tn_std["date_publication"], errors="coerce")

df_immobilier_tn_std


,url,contrat,type,prix,titre,description,surface,pieces,etage,code_postal,adresse,ville,gouvernerat,latitude,longitude,date_publication,date_scraping,standing,annee_constr,ecole,pharmacie,hopital,marche,magasin,restaurant,bus,railway,caracteristiques,images
0,https://www.immobilier.com.tn/annonce/33625/lo...,Location,Bureau,2300.00,Location : Local bureautique à louer (Marsa Ma...,Local bureautique à louer (Marsa Mall) Sidi Da...,148.00,NaN,NaN,None,rue }; Bureau 148 m² 4 Nous vous proposons à l...,Tunis,None,None,None,NaT,None,None,NaN,None,None,None,None,None,None,None,None,NaN,NaN
1,https://www.immobilier.com.tn/annonce/33645/a-...,Location,Appartement,1850.00,Location : A louer Etage de villa à Jardin d’E...,A louer Etage de villa à Jardin d’El menzah 2 ...,180.00,4.00,NaN,None,rue }; Appartement 180 m² 3 À louer – Étage de...,Ariana,None,None,None,NaT,None,None,NaN,None,None,None,None,None,None,None,None,NaN,NaN
2,https://www.immobilier.com.tn/annonce/33502/al...,Location,Bureau,10700.00,Location : AL Bureau 346m² au Lac1 - immobilie...,"Bloc B, 4ème étage, Immeuble Constance, Rue du...",346.00,NaN,NaN,None,"Bloc B, 4ème étage, Immeuble Constance, Rue du...",Tunis - Lac1,None,None,None,NaT,None,None,NaN,None,None,None,None,None,None,None,None,NaN,NaN
3,https://www.immobilier.com.tn/annonce/33651/bu...,Location,Appartement,NaN,Location : bureau haut standing - Pacha centre...,bureau haut standing - Pacha centre Pacha cent...,90.00,NaN,NaN,None,rue }; Appartement 90 m² 3 A louer un bureau h...,Tunis,None,None,None,NaT,None,None,NaN,None,None,None,None,None,None,None,None,NaN,NaN
4,https://www.immobilier.com.tn/annonce/33619/s1...,Location,Appartement,1000.00,Location : S+1 à louer - La Marsa (cité el Kha...,S+1 à louer - La Marsa (cité el Khalil) Cite E...,43.00,2.00,NaN,None,rue }; Appartement 43 m² Studio Cet appartemen...,Tunis,None,None,None,NaT,None,None,NaN,None,None,None,None,None,None,None,None,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
891,https://www.immobilier.com.tn/annonce/33637/ap...,Location,Appartement,650.00,Location : appartement deux chambres - immobi...,"appartement deux chambres Cite Ennour Jaafar, ...",82.00,3.00,NaN,None,rue }; Appartement 82 m² 2 Appartement propre ...,Ariana,None,None,None,NaT,None,None,NaN,None,None,None,None,None,None,None,None,NaN,NaN
892,https://www.immobilier.com.tn/annonce/33503/al...,Location,Bureau,8200.00,Location : AL Bureau au lac3 - immobilier.com.tn,"Bloc B, 4ème étage, Immeuble Constance, Rue du...",492.00,NaN,NaN,None,"Bloc B, 4ème étage, Immeuble Constance, Rue du...",Tunis,None,None,None,NaT,None,None,NaN,None,None,None,None,None,None,None,None,NaN,NaN
893,https://www.immobilier.com.tn/annonce/33624/au...,Location,Appartement,4200.00,Location : Au coeur de la Marsa - immobilier....,"Au coeur de la Marsa Marsa Safsaf, La Marsa, T...",80.00,3.00,NaN,None,"rue }; Appartement 80 m² 2 À louer à la Marsa,...",Tunis,None,None,None,NaT,None,None,NaN,None,None,None,None,None,None,None,None,NaN,NaN
894,https://www.immobilier.com.tn/annonce/33648/a-...,Location,Villa,NaN,Location : A louer duplex résidence Riadh Borj...,A louer duplex résidence Riadh Borj touil Raou...,124.00,NaN,NaN,None,rue }; Villa 124 m² 4 À louer – Duplex à Résid...,Ariana,None,None,None,NaT,None,None,NaN,None,None,None,None,None,None,None,None,NaN,NaN


## Annonces Immobilieres

In [20]:
data = pd.read_json('datasets/annonces_immobilieres/data.json')
df_annonces_immo = json_normalize(data.to_dict('records'))
df_annonces_immo.info()
df_annonces_immo

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 89 entries, 0 to 88
Data columns (total 17 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   source                89 non-null     object 
 1   url                   89 non-null     object 
 2   annonce_id            89 non-null     int64  
 3   transaction           89 non-null     object 
 4   type_bien             0 non-null      float64
 5   titre                 89 non-null     object 
 6   prix                  0 non-null      float64
 7   surface_m2            0 non-null      float64
 8   pieces                0 non-null      float64
 9   chambres              0 non-null      float64
 10  salles_de_bain        0 non-null      float64
 11  localisation          89 non-null     object 
 12  adresse               0 non-null      float64
 13  description           89 non-null     object 
 14  images_csv            89 non-null     object 
 15  caracteristiques_csv  89 

,source,url,annonce_id,transaction,type_bien,titre,prix,surface_m2,pieces,chambres,salles_de_bain,localisation,adresse,description,images_csv,caracteristiques_csv,details_csv
0,annonces-immobilieres.tn,https://annonces-immobilieres.tn/annonce/detai...,247,location,NaN,Opportunité Rare Ariana Essoughra. Bâtiment: V...,NaN,NaN,NaN,NaN,NaN,"Ariana, Raoued",NaN,À vendre : un bâtiment habitation commercial i...,https://annonces-immobilieres.tn/images/icon/h...,"Parking pour:2 voitures,Étages ::5 Appartement...","Jardin,Ascenseur,Cuisine équipée,À vendre : un..."
1,annonces-immobilieres.tn,https://annonces-immobilieres.tn/annonce/detai...,248,location,NaN,Location Annuelle Appartement en Zone Touristi...,NaN,NaN,NaN,NaN,NaN,"Medenine, Djerba - Midoun",NaN,Location annuelle d'un appartement en 1er étag...,https://annonces-immobilieres.tn/images/icon/h...,Location annuelle d'un appartement en:1er étag...,"Piscine,Cuisine équipée,Chauffage central,Loca..."
2,annonces-immobilieres.tn,https://annonces-immobilieres.tn/annonce/detai...,249,location,NaN,Location Annuelle Villa Neuve à la Zone Touris...,NaN,NaN,NaN,NaN,NaN,"Medenine, Djerba-Houmt Souk",NaN,Proche de toutes les commodités: Commerces & p...,https://annonces-immobilieres.tn/images/icon/h...,Elle se compose en RDC d'un espace de vie lumi...,"Jardin,Ascenseur,Cuisine équipée,Proche de tou..."
3,annonces-immobilieres.tn,https://annonces-immobilieres.tn/annonce/detai...,250,location,NaN,Location Annuelle de Grande Villa à Arkou Djer...,NaN,NaN,NaN,NaN,NaN,"Medenine, Djerba - Midoun",NaN,Une grande villa à louer à Arkou - Dans un cad...,https://annonces-immobilieres.tn/images/icon/h...,La villa se compose en RDC d'un espace de vie ...,"Garage,Jardin,Ascenseur,Cuisine équipée,Chauff..."
4,annonces-immobilieres.tn,https://annonces-immobilieres.tn/annonce/detai...,251,location,NaN,Location Annuelle Sans Meubles - Villa à Ghize...,NaN,NaN,NaN,NaN,NaN,"Medenine, Djerba-Houmt Souk",NaN,A louer à l'année - une villa neuve de style d...,https://annonces-immobilieres.tn/images/icon/h...,A louer à l'année - une villa neuve de style d...,"Jardin,Ascenseur,Cuisine équipée,A louer à l'a..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
84,annonces-immobilieres.tn,https://annonces-immobilieres.tn/annonce/detai...,255,location,NaN,Location Annuelle Villa avec Piscine à Houmt S...,NaN,NaN,NaN,NaN,NaN,"Medenine, Djerba-Houmt Souk",NaN,A louer pour longue durée une villa qui se car...,https://annonces-immobilieres.tn/images/icon/h...,Elle se compose en rez-de-chaussée d'un grand ...,"Garage,Jardin,Ascenseur,Cuisine équipée,Chauff..."
85,annonces-immobilieres.tn,https://annonces-immobilieres.tn/annonce/detai...,254,location,NaN,Maison en Rez de Chaussée à Louer à Midoun Dje...,NaN,NaN,NaN,NaN,NaN,"Medenine, Djerba - Midoun",NaN,A louer pour langue durée une maison meublée à...,https://annonces-immobilieres.tn/images/icon/h...,Une maison en rez de chaussée avec une entrée ...,"Ascenseur,Cuisine équipée,Chauffage central,A ..."
86,annonces-immobilieres.tn,https://annonces-immobilieres.tn/annonce/detai...,253,location,NaN,Location Annuelle D'un étage de Villa à la Zon...,NaN,NaN,NaN,NaN,NaN,"Medenine, Djerba-Houmt Souk",NaN,A louer un grand étage de villa avec accès pis...,https://annonces-immobilieres.tn/images/icon/h...,"L'étage se compose de:2 suites avec dressing, ...","Garage,Cuisine équipée,Chauffage central,A lou..."
87,annonces-immobilieres.tn,https://annonces-immobilieres.tn/annonce/detai...,246,location,NaN,Location Annuelle D'une Villa avec Piscine à S...,NaN,NaN,NaN,NaN,NaN,"Medenine, Djerba-Houmt Souk",NaN,Dans un cadre calme et de proximité de la plag...,https://annonces-immobilieres.tn/images/icon/h...,La villa se compose d'un beau espace de vie : ...,"Garage,Jardin,Ascenseur,Cuisine équipée,Chauff..."


In [21]:
df_annonces_immo = df_annonces_immo.replace(
    ["", " ", "NA", "N/A", "na", "null", "None", None],
    np.nan
)
null_table = (
    pd.DataFrame({
        "Null Count": df_annonces_immo.isna().sum(),
        "Null Percentage (%)": df_annonces_immo.isna().mean() * 100
    })
    .round(2)
    .sort_values(by="Null Percentage (%)", ascending=False)
)

print(null_table)

                      Null Count  Null Percentage (%)
type_bien                     89               100.00
salles_de_bain                89               100.00
pieces                        89               100.00
surface_m2                    89               100.00
prix                          89               100.00
chambres                      89               100.00
adresse                       89               100.00
details_csv                   30                33.71
source                         0                 0.00
titre                          0                 0.00
url                            0                 0.00
annonce_id                     0                 0.00
transaction                    0                 0.00
localisation                   0                 0.00
description                    0                 0.00
images_csv                     0                 0.00
caracteristiques_csv           0                 0.00


### Data Prep

In [22]:
df_annonces_immo_std = df_annonces_immo.drop(columns=[col for col in df_annonces_immo.columns if df_annonces_immo[col].isna().all()])
df_annonces_immo_std.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 89 entries, 0 to 88
Data columns (total 10 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   source                89 non-null     object
 1   url                   89 non-null     object
 2   annonce_id            89 non-null     int64 
 3   transaction           89 non-null     object
 4   titre                 89 non-null     object
 5   localisation          89 non-null     object
 6   description           89 non-null     object
 7   images_csv            89 non-null     object
 8   caracteristiques_csv  89 non-null     object
 9   details_csv           59 non-null     object
dtypes: int64(1), object(9)
memory usage: 7.1+ KB


In [23]:
def normalize_schema(df, mapping):
    # rename columns to master names
    df = df.rename(columns=mapping)

    # add missing columns
    for col in MASTER_SCHEMA:
        if col not in df.columns:
            df[col] = None

    # keep only master schema order
    df = df[MASTER_SCHEMA]

    return df

In [24]:
df_annonces_immo_std["caracteristiques_csv"]+= "\n details : " + df_annonces_immo_std["details_csv"]

print (df_annonces_immo_std["caracteristiques_csv"][0])
print ("==================================================")
print (df_annonces_immo_std["details_csv"][0])


Parking pour:2 voitures,Étages ::5 Appartements Déjà Construits – Finitions à Personnaliser,(plomberie à:50%, revêtement et peinture, y compris les escaliers),Répartition sur:3 niveaux :,Bâche à eau:3x6m pour collecter les eaux pluviales,Ariana Essoughra.:12 rue,Membre depuis:2026,Minimum:20% du prix (50,000 TND),Lun-Ven::9h-18h,©:2024 Annonces-Immobilières.tn | Rabaii Group . Tous droits réservés. Créé par
 details : Jardin,Ascenseur,Cuisine équipée,À vendre : un bâtiment habitation commercial implanté dans un quartier chic, calme, entièrement viabilisé et très recherché, à proximité immédiate d’Ariana Essoughra.,Un bien idéal pour investisseurs ou pour un projet familial haut standing, bénéficiant d’un titre foncier individuel (bleu), offrant une parfaite sécurité juridique et une excellente valeur patrimoniale.,Rez-de-chaussée : Villa Haut Standing – Totalement Achevée,Salon spacieux et lumineux,3 chambres confortables,3 salles d’eau modernes,Cuisine équipée haut de gamme,Parking po

In [25]:

mapping_annonces_immo = {
    "url": "url",
    "transaction": "contrat",
    "description": "description",
    "titre": "titre",
    "pieces": "pieces",
    "prix": "prix",
    "adresse": "adresse",
    "type_bien": "type",
    "surface_m2": "surface",
    "localisation": "ville",
    "caracteristiques_csv": "caracteristiques",
    "images_csv": "images"
}
df_annonces_immo_std = normalize_schema(df_annonces_immo_std, mapping_annonces_immo)
df_annonces_immo_std.info()
df_annonces_immo_std

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 89 entries, 0 to 88
Data columns (total 29 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   url               89 non-null     object
 1   contrat           89 non-null     object
 2   type              0 non-null      object
 3   prix              0 non-null      object
 4   titre             89 non-null     object
 5   description       89 non-null     object
 6   surface           0 non-null      object
 7   pieces            0 non-null      object
 8   etage             0 non-null      object
 9   code_postal       0 non-null      object
 10  adresse           0 non-null      object
 11  ville             89 non-null     object
 12  gouvernerat       0 non-null      object
 13  latitude          0 non-null      object
 14  longitude         0 non-null      object
 15  date_publication  0 non-null      object
 16  date_scraping     0 non-null      object
 17  standing          

,url,contrat,type,prix,titre,description,surface,pieces,etage,code_postal,adresse,ville,gouvernerat,latitude,longitude,date_publication,date_scraping,standing,annee_constr,ecole,pharmacie,hopital,marche,magasin,restaurant,bus,railway,caracteristiques,images
0,https://annonces-immobilieres.tn/annonce/detai...,location,None,None,Opportunité Rare Ariana Essoughra. Bâtiment: V...,À vendre : un bâtiment habitation commercial i...,None,None,None,None,None,"Ariana, Raoued",None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,"Parking pour:2 voitures,Étages ::5 Appartement...",https://annonces-immobilieres.tn/images/icon/h...
1,https://annonces-immobilieres.tn/annonce/detai...,location,None,None,Location Annuelle Appartement en Zone Touristi...,Location annuelle d'un appartement en 1er étag...,None,None,None,None,None,"Medenine, Djerba - Midoun",None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,Location annuelle d'un appartement en:1er étag...,https://annonces-immobilieres.tn/images/icon/h...
2,https://annonces-immobilieres.tn/annonce/detai...,location,None,None,Location Annuelle Villa Neuve à la Zone Touris...,Proche de toutes les commodités: Commerces & p...,None,None,None,None,None,"Medenine, Djerba-Houmt Souk",None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,Elle se compose en RDC d'un espace de vie lumi...,https://annonces-immobilieres.tn/images/icon/h...
3,https://annonces-immobilieres.tn/annonce/detai...,location,None,None,Location Annuelle de Grande Villa à Arkou Djer...,Une grande villa à louer à Arkou - Dans un cad...,None,None,None,None,None,"Medenine, Djerba - Midoun",None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,La villa se compose en RDC d'un espace de vie ...,https://annonces-immobilieres.tn/images/icon/h...
4,https://annonces-immobilieres.tn/annonce/detai...,location,None,None,Location Annuelle Sans Meubles - Villa à Ghize...,A louer à l'année - une villa neuve de style d...,None,None,None,None,None,"Medenine, Djerba-Houmt Souk",None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,A louer à l'année - une villa neuve de style d...,https://annonces-immobilieres.tn/images/icon/h...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
84,https://annonces-immobilieres.tn/annonce/detai...,location,None,None,Location Annuelle Villa avec Piscine à Houmt S...,A louer pour longue durée une villa qui se car...,None,None,None,None,None,"Medenine, Djerba-Houmt Souk",None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,Elle se compose en rez-de-chaussée d'un grand ...,https://annonces-immobilieres.tn/images/icon/h...
85,https://annonces-immobilieres.tn/annonce/detai...,location,None,None,Maison en Rez de Chaussée à Louer à Midoun Dje...,A louer pour langue durée une maison meublée à...,None,None,None,None,None,"Medenine, Djerba - Midoun",None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,Une maison en rez de chaussée avec une entrée ...,https://annonces-immobilieres.tn/images/icon/h...
86,https://annonces-immobilieres.tn/annonce/detai...,location,None,None,Location Annuelle D'un étage de Villa à la Zon...,A louer un grand étage de villa avec accès pis...,None,None,None,None,None,"Medenine, Djerba-Houmt Souk",None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,"L'étage se compose de:2 suites avec dressing, ...",https://annonces-immobilieres.tn/images/icon/h...
87,https://annonces-immobilieres.tn/annonce/detai...,location,None,None,Location Annuelle D'une Villa avec Piscine à S...,Dans un cadre calme et de proximité de la plag...,None,None,None,None,None,"Medenine, Djerba-Houmt Souk",None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,La villa se compose d'un beau espace de vie : ...,https://annonces-immobilieres.tn/images/icon/h...


In [26]:
df_annonces_immo_std["description"] = np.where(
    df_annonces_immo_std["caracteristiques"].notna(),
    df_annonces_immo_std["description"].fillna("") +
    "\nCaracteristiques: " +
    df_annonces_immo_std["caracteristiques"],
    df_annonces_immo_std["description"]
)

df_annonces_immo_std["caracteristiques"] = np.nan
df_annonces_immo_std

,url,contrat,type,prix,titre,description,surface,pieces,etage,code_postal,adresse,ville,gouvernerat,latitude,longitude,date_publication,date_scraping,standing,annee_constr,ecole,pharmacie,hopital,marche,magasin,restaurant,bus,railway,caracteristiques,images
0,https://annonces-immobilieres.tn/annonce/detai...,location,None,None,Opportunité Rare Ariana Essoughra. Bâtiment: V...,À vendre : un bâtiment habitation commercial i...,None,None,None,None,None,"Ariana, Raoued",None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,NaN,https://annonces-immobilieres.tn/images/icon/h...
1,https://annonces-immobilieres.tn/annonce/detai...,location,None,None,Location Annuelle Appartement en Zone Touristi...,Location annuelle d'un appartement en 1er étag...,None,None,None,None,None,"Medenine, Djerba - Midoun",None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,NaN,https://annonces-immobilieres.tn/images/icon/h...
2,https://annonces-immobilieres.tn/annonce/detai...,location,None,None,Location Annuelle Villa Neuve à la Zone Touris...,Proche de toutes les commodités: Commerces & p...,None,None,None,None,None,"Medenine, Djerba-Houmt Souk",None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,NaN,https://annonces-immobilieres.tn/images/icon/h...
3,https://annonces-immobilieres.tn/annonce/detai...,location,None,None,Location Annuelle de Grande Villa à Arkou Djer...,Une grande villa à louer à Arkou - Dans un cad...,None,None,None,None,None,"Medenine, Djerba - Midoun",None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,NaN,https://annonces-immobilieres.tn/images/icon/h...
4,https://annonces-immobilieres.tn/annonce/detai...,location,None,None,Location Annuelle Sans Meubles - Villa à Ghize...,A louer à l'année - une villa neuve de style d...,None,None,None,None,None,"Medenine, Djerba-Houmt Souk",None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,NaN,https://annonces-immobilieres.tn/images/icon/h...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
84,https://annonces-immobilieres.tn/annonce/detai...,location,None,None,Location Annuelle Villa avec Piscine à Houmt S...,A louer pour longue durée une villa qui se car...,None,None,None,None,None,"Medenine, Djerba-Houmt Souk",None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,NaN,https://annonces-immobilieres.tn/images/icon/h...
85,https://annonces-immobilieres.tn/annonce/detai...,location,None,None,Maison en Rez de Chaussée à Louer à Midoun Dje...,A louer pour langue durée une maison meublée à...,None,None,None,None,None,"Medenine, Djerba - Midoun",None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,NaN,https://annonces-immobilieres.tn/images/icon/h...
86,https://annonces-immobilieres.tn/annonce/detai...,location,None,None,Location Annuelle D'un étage de Villa à la Zon...,A louer un grand étage de villa avec accès pis...,None,None,None,None,None,"Medenine, Djerba-Houmt Souk",None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,NaN,https://annonces-immobilieres.tn/images/icon/h...
87,https://annonces-immobilieres.tn/annonce/detai...,location,None,None,Location Annuelle D'une Villa avec Piscine à S...,Dans un cadre calme et de proximité de la plag...,None,None,None,None,None,"Medenine, Djerba-Houmt Souk",None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,NaN,https://annonces-immobilieres.tn/images/icon/h...


#### Data type conversion

In [27]:
df_annonces_immo_std["prix"] = (
    df_annonces_immo_std["prix"]
    .astype(str)
    .str.replace(r"[^\d.]", "", regex=True)  # keep only numbers
)

df_annonces_immo_std["prix"] = pd.to_numeric(df_annonces_immo_std["prix"], errors="coerce")

df_annonces_immo_std["surface"] = pd.to_numeric(df_annonces_immo_std["surface"], errors="coerce")

df_annonces_immo_std["etage"] = pd.to_numeric(df_annonces_immo_std["etage"], errors="coerce")
df_annonces_immo_std["annee_constr"] = pd.to_numeric(df_annonces_immo_std["annee_constr"], errors="coerce")

df_immobilier_tn_std["pieces"] = pd.to_numeric(df_immobilier_tn_std["pieces"], errors="coerce")

df_immobilier_tn_std["date_publication"] = pd.to_datetime(df_immobilier_tn_std["date_publication"], errors="coerce")

df_immobilier_tn_std


,url,contrat,type,prix,titre,description,surface,pieces,etage,code_postal,adresse,ville,gouvernerat,latitude,longitude,date_publication,date_scraping,standing,annee_constr,ecole,pharmacie,hopital,marche,magasin,restaurant,bus,railway,caracteristiques,images
0,https://www.immobilier.com.tn/annonce/33625/lo...,Location,Bureau,2300.00,Location : Local bureautique à louer (Marsa Ma...,Local bureautique à louer (Marsa Mall) Sidi Da...,148.00,NaN,NaN,None,rue }; Bureau 148 m² 4 Nous vous proposons à l...,Tunis,None,None,None,NaT,None,None,NaN,None,None,None,None,None,None,None,None,NaN,NaN
1,https://www.immobilier.com.tn/annonce/33645/a-...,Location,Appartement,1850.00,Location : A louer Etage de villa à Jardin d’E...,A louer Etage de villa à Jardin d’El menzah 2 ...,180.00,4.00,NaN,None,rue }; Appartement 180 m² 3 À louer – Étage de...,Ariana,None,None,None,NaT,None,None,NaN,None,None,None,None,None,None,None,None,NaN,NaN
2,https://www.immobilier.com.tn/annonce/33502/al...,Location,Bureau,10700.00,Location : AL Bureau 346m² au Lac1 - immobilie...,"Bloc B, 4ème étage, Immeuble Constance, Rue du...",346.00,NaN,NaN,None,"Bloc B, 4ème étage, Immeuble Constance, Rue du...",Tunis - Lac1,None,None,None,NaT,None,None,NaN,None,None,None,None,None,None,None,None,NaN,NaN
3,https://www.immobilier.com.tn/annonce/33651/bu...,Location,Appartement,NaN,Location : bureau haut standing - Pacha centre...,bureau haut standing - Pacha centre Pacha cent...,90.00,NaN,NaN,None,rue }; Appartement 90 m² 3 A louer un bureau h...,Tunis,None,None,None,NaT,None,None,NaN,None,None,None,None,None,None,None,None,NaN,NaN
4,https://www.immobilier.com.tn/annonce/33619/s1...,Location,Appartement,1000.00,Location : S+1 à louer - La Marsa (cité el Kha...,S+1 à louer - La Marsa (cité el Khalil) Cite E...,43.00,2.00,NaN,None,rue }; Appartement 43 m² Studio Cet appartemen...,Tunis,None,None,None,NaT,None,None,NaN,None,None,None,None,None,None,None,None,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
891,https://www.immobilier.com.tn/annonce/33637/ap...,Location,Appartement,650.00,Location : appartement deux chambres - immobi...,"appartement deux chambres Cite Ennour Jaafar, ...",82.00,3.00,NaN,None,rue }; Appartement 82 m² 2 Appartement propre ...,Ariana,None,None,None,NaT,None,None,NaN,None,None,None,None,None,None,None,None,NaN,NaN
892,https://www.immobilier.com.tn/annonce/33503/al...,Location,Bureau,8200.00,Location : AL Bureau au lac3 - immobilier.com.tn,"Bloc B, 4ème étage, Immeuble Constance, Rue du...",492.00,NaN,NaN,None,"Bloc B, 4ème étage, Immeuble Constance, Rue du...",Tunis,None,None,None,NaT,None,None,NaN,None,None,None,None,None,None,None,None,NaN,NaN
893,https://www.immobilier.com.tn/annonce/33624/au...,Location,Appartement,4200.00,Location : Au coeur de la Marsa - immobilier....,"Au coeur de la Marsa Marsa Safsaf, La Marsa, T...",80.00,3.00,NaN,None,"rue }; Appartement 80 m² 2 À louer à la Marsa,...",Tunis,None,None,None,NaT,None,None,NaN,None,None,None,None,None,None,None,None,NaN,NaN
894,https://www.immobilier.com.tn/annonce/33648/a-...,Location,Villa,NaN,Location : A louer duplex résidence Riadh Borj...,A louer duplex résidence Riadh Borj touil Raou...,124.00,NaN,NaN,None,rue }; Villa 124 m² 4 À louer – Duplex à Résid...,Ariana,None,None,None,NaT,None,None,NaN,None,None,None,None,None,None,None,None,NaN,NaN


## Tunisie Annonce

In [28]:
data = pd.read_json('datasets/tunisie_annonce/data.json')
df_tunisie_annonce = json_normalize(data.to_dict('records'))
df_tunisie_annonce.info()
df_tunisie_annonce

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15172 entries, 0 to 15171
Data columns (total 36 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   location             15172 non-null  object 
 1   governorate          15172 non-null  object 
 2   delegation           15172 non-null  object 
 3   locality             15172 non-null  object 
 4   property_type        15172 non-null  object 
 5   property_subtype     15172 non-null  object 
 6   title                15172 non-null  object 
 7   description_preview  15156 non-null  object 
 8   price_tnd            15124 non-null  float64
 9   price_raw            15172 non-null  object 
 10  date_posted          15172 non-null  object 
 11  detail_url           15172 non-null  object 
 12  listing_page_url     15172 non-null  object 
 13  reference_number     15172 non-null  int64  
 14  title_full           15172 non-null  object 
 15  category_path        15172 non-null 

,location,governorate,delegation,locality,property_type,property_subtype,title,description_preview,price_tnd,price_raw,date_posted,detail_url,listing_page_url,reference_number,title_full,category_path,category_main,category_type,category_subtype,location_path,country,region,city,locality_detail,surface_raw,surface_m2,price_detail,price_eur,price_usd,price_cad,price_dzd,price_mad,description_full,date_inserted,date_modified,address
0,Ariana,Ariana,Ariana Ville,Ariana,Location,App. 4 pièc,S plus 3 a l'ariana,S plus 3 a l ariana wb4420,1200.00,1 200,15/02/2026,http://www.tunisie-annonce.com/Details_Annonce...,http://www.tunisie-annonce.com/AnnoncesImmobil...,3450753,[Réf:3450753] S plus 3 a l'ariana wb4420,Offres > Location > Appart. 4 pièces,Offres,Location,Appart. 4 pièces,Tunisie > Ariana > Ariana Ville > Ariana,Tunisie,Ariana,Ariana Ville,Ariana,120 m²,120.00,1 200 Dinar Tunisien (TND),367,422,547,50 092,4 001,l'agence immo contact met en location a l'aria...,15/02/2026,15/02/2026,NaN
1,Ariana,Ariana,Ariana Ville,Ariana,Location,App. 3 pièc,Appartement borj louzir,Appartement borj louzir,550.00,550,15/02/2026,http://www.tunisie-annonce.com/Details_Annonce...,http://www.tunisie-annonce.com/AnnoncesImmobil...,3450755,[Réf:3450755] Appartement borj louzir,Offres > Location > Appart. 3 pièces,Offres,Location,Appart. 3 pièces,Tunisie > Ariana > Ariana Ville > Ariana,Tunisie,Ariana,Ariana Ville,Ariana,80 m²,80.00,550 Dinar Tunisien (TND),168,193,251,22 959,1 834,belle appartement au rez-de-chaussé. situé à b...,15/02/2026,15/02/2026,borj louzir
2,El Omrane Super,Tunis,El Omrane Superieur,El Omrane Superieur,Location,App. 1 pièc,Studio pour fille ou cou,Studio pour fille ou coupe a 100m arrêt de métro,350.00,350,15/02/2026,http://www.tunisie-annonce.com/Details_Annonce...,http://www.tunisie-annonce.com/AnnoncesImmobil...,3398506,[Réf:3398506] Studio pour fille ou coupe a 10...,Offres > Location > Appart. 1 pièce,Offres,Location,Appart. 1 pièce,Tunisie > Tunis > El Omrane Superieur > El Omr...,Tunisie,Tunis,El Omrane Superieur,El Omrane Superieur,34 m²,34.00,350 Dinar Tunisien (TND),107,123,159,14 610,1 167,"pour filles ou jeune coupe marié, un petit s+1...",29/04/2025,15/02/2026,a 100m arrêt de métro ettadhamen
3,Berge Du Lac,Tunis,La Marsa,Berge Du Lac,Bureaux & Commer,Surfaces,Local commercial 2000 d,Local commercial 2000 d lac 1,2000.00,2 000,15/02/2026,http://www.tunisie-annonce.com/Details_Annonce...,http://www.tunisie-annonce.com/AnnoncesImmobil...,3447030,[Réf:3447030] Local commercial 2000 d lac 1,Offres > Bureaux & Commerces > Surfaces,Offres,Bureaux & Commerces,Surfaces,Tunisie > Tunis > La Marsa > Berge Du Lac,Tunisie,Tunis,La Marsa,Berge Du Lac,107 m²,107.00,2 000 Dinar Tunisien (TND),612,703,911,83 487,6 668,les berges du lac 1 opportunité à saisir lo...,20/01/2026,15/02/2026,les berges du lac 1
4,Hammamet,Nabeul,Hammamet,Hammamet,Vente,Maisons,Villa manhattan,Villa manhattan,1050000.00,1 050 000,15/02/2026,http://www.tunisie-annonce.com/Details_Annonce...,http://www.tunisie-annonce.com/AnnoncesImmobil...,3399447,[Réf:3399447] Villa manhattan,Offres > Vente > Maisons,Offres,Vente,Maisons,Tunisie > Nabeul > Hammamet > Hammamet,Tunisie,Nabeul,Hammamet,Hammamet,420 m²,420.00,1 050 000 Dinar Tunisien (TND),321 101,369 009,478 280,43 830 596,3 500 835,à vendre chez green immobilier une demeure bie...,04/05/2025,15/02/2026,hammamet nord
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15167,Hammamet,Nabeul,Hammamet,Hammamet,Vente,Maisons,V536 villa suisse 2 hamm,V536 villa suisse 2 hammamet nord villa,750000.00,750 000,31/10/2025,http://www.tunisie-annonce.com/Details_Annonce...,http://www.tunisie-annonce.com/AnnoncesImmobil...,3433031,[Réf:3433031] V536 villa suisse 2 hammamet nor...,Offres > Vente > Maisons,Offres,Vente,Maisons,Tunisie > Nabeul > Hammamet > Hammamet,Tunisie,Nabeul,Hammamet,Hammamet,210 m²,210.00,750 000 Dinar 

In [29]:
df_tunisie_annonce = df_tunisie_annonce.replace(
    ["", " ", "NA", "N/A", "na", "null", "None", None],
    np.nan
)
null_table = (
    pd.DataFrame({
        "Null Count": df_tunisie_annonce.isna().sum(),
        "Null Percentage (%)": df_tunisie_annonce.isna().mean() * 100
    })
    .round(2)
    .sort_values(by="Null Percentage (%)", ascending=False)
)

print(null_table)

                     Null Count  Null Percentage (%)
address                    5729                37.76
surface_m2                  916                 6.04
surface_raw                 916                 6.04
price_tnd                    48                 0.32
description_preview          24                 0.16
delegation                    0                 0.00
governorate                   0                 0.00
location                      0                 0.00
locality                      0                 0.00
title                         0                 0.00
property_subtype              0                 0.00
property_type                 0                 0.00
listing_page_url              0                 0.00
reference_number              0                 0.00
title_full                    0                 0.00
category_path                 0                 0.00
category_main                 0                 0.00
price_raw                     0               

### Data Prep

In [30]:
df_tunisie_annonce_std = df_tunisie_annonce.drop(columns=[col for col in df_tunisie_annonce.columns if df_tunisie_annonce[col].isna().all()])
df_tunisie_annonce_std.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15172 entries, 0 to 15171
Data columns (total 36 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   location             15172 non-null  object 
 1   governorate          15172 non-null  object 
 2   delegation           15172 non-null  object 
 3   locality             15172 non-null  object 
 4   property_type        15172 non-null  object 
 5   property_subtype     15172 non-null  object 
 6   title                15172 non-null  object 
 7   description_preview  15148 non-null  object 
 8   price_tnd            15124 non-null  float64
 9   price_raw            15172 non-null  object 
 10  date_posted          15172 non-null  object 
 11  detail_url           15172 non-null  object 
 12  listing_page_url     15172 non-null  object 
 13  reference_number     15172 non-null  int64  
 14  title_full           15172 non-null  object 
 15  category_path        15172 non-null 

In [31]:
def normalize_schema(df, mapping):
    # rename columns to master names
    df = df.rename(columns=mapping)

    # add missing columns
    for col in MASTER_SCHEMA:
        if col not in df.columns:
            df[col] = None

    # keep only master schema order
    df = df[MASTER_SCHEMA]

    return df

In [32]:
mapping_tunisie_annonce = {
    "title_full": "titre",
    "description_full": "description",
    "locality_detail": "adresse",
    "city": "ville",
    "region": "gouvernerat",
    "detail_url": "url",
    "price_tnd": "prix",
    "surface_m2": "surface",
    "property_subtype" : "type",
    "date_posted": "date_publication",
    "category_type": "contrat",
}
df_tunisie_annonce_std = normalize_schema(df_tunisie_annonce_std, mapping_tunisie_annonce)
df_tunisie_annonce_std.info()
df_tunisie_annonce_std

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15172 entries, 0 to 15171
Data columns (total 29 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   url               15172 non-null  object 
 1   contrat           15172 non-null  object 
 2   type              15172 non-null  object 
 3   prix              15124 non-null  float64
 4   titre             15172 non-null  object 
 5   description       15172 non-null  object 
 6   surface           14256 non-null  float64
 7   pieces            0 non-null      object 
 8   etage             0 non-null      object 
 9   code_postal       0 non-null      object 
 10  adresse           15172 non-null  object 
 11  ville             15172 non-null  object 
 12  gouvernerat       15172 non-null  object 
 13  latitude          0 non-null      object 
 14  longitude         0 non-null      object 
 15  date_publication  15172 non-null  object 
 16  date_scraping     0 non-null      object

,url,contrat,type,prix,titre,description,surface,pieces,etage,code_postal,adresse,ville,gouvernerat,latitude,longitude,date_publication,date_scraping,standing,annee_constr,ecole,pharmacie,hopital,marche,magasin,restaurant,bus,railway,caracteristiques,images
0,http://www.tunisie-annonce.com/Details_Annonce...,Location,App. 4 pièc,1200.00,[Réf:3450753] S plus 3 a l'ariana wb4420,l'agence immo contact met en location a l'aria...,120.00,None,None,None,Ariana,Ariana Ville,Ariana,None,None,15/02/2026,None,None,None,None,None,None,None,None,None,None,None,None,None
1,http://www.tunisie-annonce.com/Details_Annonce...,Location,App. 3 pièc,550.00,[Réf:3450755] Appartement borj louzir,belle appartement au rez-de-chaussé. situé à b...,80.00,None,None,None,Ariana,Ariana Ville,Ariana,None,None,15/02/2026,None,None,None,None,None,None,None,None,None,None,None,None,None
2,http://www.tunisie-annonce.com/Details_Annonce...,Location,App. 1 pièc,350.00,[Réf:3398506] Studio pour fille ou coupe a 10...,"pour filles ou jeune coupe marié, un petit s+1...",34.00,None,None,None,El Omrane Superieur,El Omrane Superieur,Tunis,None,None,15/02/2026,None,None,None,None,None,None,None,None,None,None,None,None,None
3,http://www.tunisie-annonce.com/Details_Annonce...,Bureaux & Commerces,Surfaces,2000.00,[Réf:3447030] Local commercial 2000 d lac 1,les berges du lac 1 opportunité à saisir lo...,107.00,None,None,None,Berge Du Lac,La Marsa,Tunis,None,None,15/02/2026,None,None,None,None,None,None,None,None,None,None,None,None,None
4,http://www.tunisie-annonce.com/Details_Annonce...,Vente,Maisons,1050000.00,[Réf:3399447] Villa manhattan,à vendre chez green immobilier une demeure bie...,420.00,None,None,None,Hammamet,Hammamet,Nabeul,None,None,15/02/2026,None,None,None,None,None,None,None,None,None,None,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15167,http://www.tunisie-annonce.com/Details_Annonce...,Vente,Maisons,750000.00,[Réf:3433031] V536 villa suisse 2 hammamet nor...,v536 villa suisse 2\na vendre chez l’agence sm...,210.00,None,None,None,Hammamet,Hammamet,Nabeul,None,None,31/10/2025,None,None,None,None,None,None,None,None,None,None,None,None,None
15168,http://www.tunisie-annonce.com/Details_Annonce...,Location,App. 3 pièc,2300.00,[Réf:3427265] Un charmant apparemment s2 au ja...,un bel appartement s2 situé dans une résidence...,1.00,None,None,None,Jardins de Carthage,Ain Zaghouan,Tunis,None,None,31/10/2025,None,None,None,None,None,None,None,None,None,None,None,None,None
15169,http://www.tunisie-annonce.com/Details_Annonce...,Location,App. 3 pièc,600.00,[Réf:3433145] Reez de chaussez s2 à b jezira tn,a louer un appart s2 situé à tunis centre vill...,75.00,None,None,None,Bab El Jazira,Bab Bhar,Tunis,None,None,01/11/2025,None,None,None,None,None,None,None,None,None,None,None,None,None
15170,http://www.tunisie-annonce.com/Details_Annonce...,Location,App. 3 pièc,600.00,[Réf:3433144] S2 au centre ville de tunis,a louer un appart s2 situé à tunis centre vill...,75.00,None,None,None,Bab El Jazira,Bab Bhar,Tunis,None,None,01/11/2025,None,None,None,None,None,None,None,None,None,None,None,None,None


In [33]:
df_tunisie_annonce_std["pieces"] = df_tunisie_annonce_std["type"].str.extract(r"(\d+)")  # extract digits

In [34]:
df_tunisie_annonce_std["type"] = df_tunisie_annonce_std["type"].str.split().str[0]

In [35]:
df_tunisie_annonce_std

,url,contrat,type,prix,titre,description,surface,pieces,etage,code_postal,adresse,ville,gouvernerat,latitude,longitude,date_publication,date_scraping,standing,annee_constr,ecole,pharmacie,hopital,marche,magasin,restaurant,bus,railway,caracteristiques,images
0,http://www.tunisie-annonce.com/Details_Annonce...,Location,App.,1200.00,[Réf:3450753] S plus 3 a l'ariana wb4420,l'agence immo contact met en location a l'aria...,120.00,4,None,None,Ariana,Ariana Ville,Ariana,None,None,15/02/2026,None,None,None,None,None,None,None,None,None,None,None,None,None
1,http://www.tunisie-annonce.com/Details_Annonce...,Location,App.,550.00,[Réf:3450755] Appartement borj louzir,belle appartement au rez-de-chaussé. situé à b...,80.00,3,None,None,Ariana,Ariana Ville,Ariana,None,None,15/02/2026,None,None,None,None,None,None,None,None,None,None,None,None,None
2,http://www.tunisie-annonce.com/Details_Annonce...,Location,App.,350.00,[Réf:3398506] Studio pour fille ou coupe a 10...,"pour filles ou jeune coupe marié, un petit s+1...",34.00,1,None,None,El Omrane Superieur,El Omrane Superieur,Tunis,None,None,15/02/2026,None,None,None,None,None,None,None,None,None,None,None,None,None
3,http://www.tunisie-annonce.com/Details_Annonce...,Bureaux & Commerces,Surfaces,2000.00,[Réf:3447030] Local commercial 2000 d lac 1,les berges du lac 1 opportunité à saisir lo...,107.00,NaN,None,None,Berge Du Lac,La Marsa,Tunis,None,None,15/02/2026,None,None,None,None,None,None,None,None,None,None,None,None,None
4,http://www.tunisie-annonce.com/Details_Annonce...,Vente,Maisons,1050000.00,[Réf:3399447] Villa manhattan,à vendre chez green immobilier une demeure bie...,420.00,NaN,None,None,Hammamet,Hammamet,Nabeul,None,None,15/02/2026,None,None,None,None,None,None,None,None,None,None,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15167,http://www.tunisie-annonce.com/Details_Annonce...,Vente,Maisons,750000.00,[Réf:3433031] V536 villa suisse 2 hammamet nor...,v536 villa suisse 2\na vendre chez l’agence sm...,210.00,NaN,None,None,Hammamet,Hammamet,Nabeul,None,None,31/10/2025,None,None,None,None,None,None,None,None,None,None,None,None,None
15168,http://www.tunisie-annonce.com/Details_Annonce...,Location,App.,2300.00,[Réf:3427265] Un charmant apparemment s2 au ja...,un bel appartement s2 situé dans une résidence...,1.00,3,None,None,Jardins de Carthage,Ain Zaghouan,Tunis,None,None,31/10/2025,None,None,None,None,None,None,None,None,None,None,None,None,None
15169,http://www.tunisie-annonce.com/Details_Annonce...,Location,App.,600.00,[Réf:3433145] Reez de chaussez s2 à b jezira tn,a louer un appart s2 situé à tunis centre vill...,75.00,3,None,None,Bab El Jazira,Bab Bhar,Tunis,None,None,01/11/2025,None,None,None,None,None,None,None,None,None,None,None,None,None
15170,http://www.tunisie-annonce.com/Details_Annonce...,Location,App.,600.00,[Réf:3433144] S2 au centre ville de tunis,a louer un appart s2 situé à tunis centre vill...,75.00,3,None,None,Bab El Jazira,Bab Bhar,Tunis,None,None,01/11/2025,None,None,None,None,None,None,None,None,None,None,None,None,None


#### Data type conversion

In [36]:
df_tunisie_annonce_std["prix"] = (
    df_tunisie_annonce_std["prix"]
    .astype(str)
    .str.replace(r"[^\d.]", "", regex=True)  # keep only numbers
)

df_tunisie_annonce_std["prix"] = pd.to_numeric(df_tunisie_annonce_std["prix"], errors="coerce")

df_tunisie_annonce_std["surface"] = pd.to_numeric(df_tunisie_annonce_std["surface"], errors="coerce")

df_tunisie_annonce_std["etage"] = pd.to_numeric(df_tunisie_annonce_std["etage"], errors="coerce")
df_tunisie_annonce_std["annee_constr"] = pd.to_numeric(df_tunisie_annonce_std["annee_constr"], errors="coerce")

df_tunisie_annonce_std["pieces"] = pd.to_numeric(df_tunisie_annonce_std["pieces"], errors="coerce")

df_tunisie_annonce_std["date_publication"] = pd.to_datetime(df_tunisie_annonce_std["date_publication"], errors="coerce")

df_tunisie_annonce_std


,url,contrat,type,prix,titre,description,surface,pieces,etage,code_postal,adresse,ville,gouvernerat,latitude,longitude,date_publication,date_scraping,standing,annee_constr,ecole,pharmacie,hopital,marche,magasin,restaurant,bus,railway,caracteristiques,images
0,http://www.tunisie-annonce.com/Details_Annonce...,Location,App.,1200.00,[Réf:3450753] S plus 3 a l'ariana wb4420,l'agence immo contact met en location a l'aria...,120.00,4.00,NaN,None,Ariana,Ariana Ville,Ariana,None,None,2026-02-15,None,None,NaN,None,None,None,None,None,None,None,None,None,None
1,http://www.tunisie-annonce.com/Details_Annonce...,Location,App.,550.00,[Réf:3450755] Appartement borj louzir,belle appartement au rez-de-chaussé. situé à b...,80.00,3.00,NaN,None,Ariana,Ariana Ville,Ariana,None,None,2026-02-15,None,None,NaN,None,None,None,None,None,None,None,None,None,None
2,http://www.tunisie-annonce.com/Details_Annonce...,Location,App.,350.00,[Réf:3398506] Studio pour fille ou coupe a 10...,"pour filles ou jeune coupe marié, un petit s+1...",34.00,1.00,NaN,None,El Omrane Superieur,El Omrane Superieur,Tunis,None,None,2026-02-15,None,None,NaN,None,None,None,None,None,None,None,None,None,None
3,http://www.tunisie-annonce.com/Details_Annonce...,Bureaux & Commerces,Surfaces,2000.00,[Réf:3447030] Local commercial 2000 d lac 1,les berges du lac 1 opportunité à saisir lo...,107.00,NaN,NaN,None,Berge Du Lac,La Marsa,Tunis,None,None,2026-02-15,None,None,NaN,None,None,None,None,None,None,None,None,None,None
4,http://www.tunisie-annonce.com/Details_Annonce...,Vente,Maisons,1050000.00,[Réf:3399447] Villa manhattan,à vendre chez green immobilier une demeure bie...,420.00,NaN,NaN,None,Hammamet,Hammamet,Nabeul,None,None,2026-02-15,None,None,NaN,None,None,None,None,None,None,None,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15167,http://www.tunisie-annonce.com/Details_Annonce...,Vente,Maisons,750000.00,[Réf:3433031] V536 villa suisse 2 hammamet nor...,v536 villa suisse 2\na vendre chez l’agence sm...,210.00,NaN,NaN,None,Hammamet,Hammamet,Nabeul,None,None,2025-10-31,None,None,NaN,None,None,None,None,None,None,None,None,None,None
15168,http://www.tunisie-annonce.com/Details_Annonce...,Location,App.,2300.00,[Réf:3427265] Un charmant apparemment s2 au ja...,un bel appartement s2 situé dans une résidence...,1.00,3.00,NaN,None,Jardins de Carthage,Ain Zaghouan,Tunis,None,None,2025-10-31,None,None,NaN,None,None,None,None,None,None,None,None,None,None
15169,http://www.tunisie-annonce.com/Details_Annonce...,Location,App.,600.00,[Réf:3433145] Reez de chaussez s2 à b jezira tn,a louer un appart s2 situé à tunis centre vill...,75.00,3.00,NaN,None,Bab El Jazira,Bab Bhar,Tunis,None,None,2025-11-01,None,None,NaN,None,None,None,None,None,None,None,None,None,None
15170,http://www.tunisie-annonce.com/Details_Annonce...,Location,App.,600.00,[Réf:3433144] S2 au centre ville de tunis,a louer un appart s2 situé à tunis centre vill...,75.00,3.00,NaN,None,Bab El Jazira,Bab Bhar,Tunis,None,None,2025-11-01,None,None,NaN,None,None,None,None,None,None,None,None,None,None


## Bigdatis

In [39]:
import pandas as pd
import json
from pandas import json_normalize

# 1. Process the First File (finale_bigdatis_details.json)
# Using read_json and then to_dict('records') is a safe way to handle nested objects
data_details = pd.read_json('datasets/Bigdatis/finale_bigdatis_details.json')
df_details = json_normalize(data_details.to_dict('records'))

# 2. Process the Second File (bigdatis_properties.json)
# Point directly to the 'properties' list inside the raw JSON
with open('datasets/bigdatis_properties.json', 'r') as f:
    raw_data_props = json.load(f)

df_props_only = json_normalize(raw_data_props['properties'])

# 3. Append the two lists into one DataFrame
# 'ignore_index=True' ensures the new DF has a clean 0 to N index
df_bigdatis = pd.concat([df_details, df_props_only], ignore_index=True)

# 4. Final inspection
print(f"Total properties combined: {len(df_bigdatis)}")
df_bigdatis.info()
display(df_bigdatis.head())

Total properties combined: 130826
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 130826 entries, 0 to 130825
Data columns (total 31 columns):
 #   Column                      Non-Null Count   Dtype  
---  ------                      --------------   -----  
 0   id                          130826 non-null  int64  
 1   idsAlt                      130826 non-null  object 
 2   sources                     130826 non-null  object 
 3   title                       130826 non-null  object 
 4   description                 130826 non-null  object 
 5   price                       104541 non-null  float64
 6   area                        68567 non-null   float64
 7   flags                       130826 non-null  object 
 8   thumbnailUrl                103208 non-null  object 
 9   images                      130826 non-null  object 
 10  imageUrls                   130826 non-null  object 
 11  locationId                  117431 non-null  float64
 12  sellerTypes                 130826 non

,id,idsAlt,sources,title,description,price,area,flags,thumbnailUrl,images,imageUrls,locationId,sellerTypes,contacts,activeContactsCount,adsCount,activeAdsCount,sourcesCount,activeSourcesCount,firstSeenAt,createdAt,modifiedAt,priceDroppedAt,timestamp,priceTimestamp,comments,commentsCount,properties.transactionType,properties.sellerType,properties.propertyType,properties.typology
0,921274,[],"[{'sourceId': 2, 'lastModified': 1771364245, '...",Location studio manzah 9 bien équipé,Un large studio de 60 métres carrés avec cuisi...,800.00,60.00,[],56202/28101387_f3327061e9695940.jpg,[{'url': 'https://storage.googleapis.com/tayar...,[https://storage.googleapis.com/tayara-migrati...,5207.00,[private],"[{'sellerType': 'private', 'contactName': 'Moh...",1,2,1,1,1,2024-12-30 23:00:20,2024-12-30 22:53:14,2026-02-17 21:37:25,2024-12-30 22:53:14,2026-02-17 21:37:25,2024-12-30 22:53:14,[],0,rental,private,flat,s+1
1,1211799,[],"[{'sourceId': 5, 'lastModified': 1771364733, '...",S2 en location,Cet appartement jamais habité est situé au rez...,850.00,85.00,[new],https://www.mubawab-media.com/ad/8/269/642F/h/...,[{'url': 'https://www.mubawab-media.com/ad/8/2...,[https://www.mubawab-media.com/ad/8/269/642F/h...,4901.00,[agency],"[{'sellerType': 'agency', 'contactName': 'MFK ...",1,3,3,3,3,2025-12-09 19:15:04,2025-12-09 19:02:24,2026-02-17 21:45:33,2025-12-09 19:02:24,2026-02-17 21:45:33,2025-12-09 19:02:24,[],0,rental,agency,flat,s+2
2,1186745,[],"[{'sourceId': 5, 'lastModified': 1771361145, '...",ENNASR 2 à louer spacieux appt S3 coté collège,ENNASR 2 ; à Louer spacieux Appartement S3 dan...,NaN,170.00,[],https://www.mubawab-media.com/ad/8/237/072F/h/...,[{'url': 'https://www.mubawab-media.com/ad/8/2...,[https://www.mubawab-media.com/ad/8/237/072F/h...,300.00,[agency],"[{'sellerType': 'agency', 'contactName': 'Immo...",1,2,2,2,2,2025-10-18 08:30:36,2025-10-18 08:30:33,2026-02-17 20:45:45,2025-10-18 08:30:33,2026-02-17 20:45:45,2025-10-18 08:30:33,[],0,rental,agency,flat,s+3
3,1234370,[],"[{'sourceId': 5, 'lastModified': 1771361145, '...",Un appartement S3 vide a la Marsa,1Step2House vous propose à la location un appa...,2700.00,140.00,[],56075/28037868_c8d8d9eb61f7f3cc.jpg,[{'url': 'https://www.mubawab-media.com/ad/8/2...,[https://www.mubawab-media.com/ad/8/285/913F/h...,5220.00,[agency],"[{'sellerType': 'agency', 'contactName': '1Ste...",1,1,1,1,1,2026-01-13 20:00:21,2026-01-13 20:00:10,2026-02-17 20:45:45,2026-01-13 20:00:10,2026-02-17 20:45:45,2026-01-13 20:00:10,[],0,rental,agency,flat,s+3
4,1234371,[],"[{'sourceId': 5, 'lastModified': 1771361145, '...",Un appartement S4 vide à la Marsa,1Step2House vous propose à la location un appa...,4500.00,200.00,[],56075/28037869_30eac04000000000.jpg,[{'url': 'https://www.mubawab-media.com/ad/8/2...,[https://www.mubawab-media.com/ad/8/285/909F/h...,5220.00,[agency],"[{'sellerType': 'agency', 'contactName': '1Ste...",1,1,1,1,1,2026-01-13 20:00:21,2026-01-13 20:00:10,2026-02-17 20:45:45,2026-01-13 20:00:10,2026-02-17 20:45:45,2026-01-13 20:00:10,[],0,rental,agency,flat,s+4


In [40]:
df_bigdatis = df_bigdatis.replace(
    ["", " ", "NA", "N/A", "na", "null", "None", None],
    np.nan
)
null_table = (
    pd.DataFrame({
        "Null Count": df_bigdatis.isna().sum(),
        "Null Percentage (%)": df_bigdatis.isna().mean() * 100
    })
    .round(2)
    .sort_values(by="Null Percentage (%)", ascending=False)
)

print(null_table)

                            Null Count  Null Percentage (%)
area                             62259                47.59
createdAt                        49593                37.91
priceDroppedAt                   46172                35.29
priceTimestamp                   46172                35.29
properties.typology              37442                28.62
thumbnailUrl                     27618                21.11
price                            26285                20.09
locationId                       13395                10.24
modifiedAt                       11337                 8.67
timestamp                        11337                 8.67
properties.transactionType        5100                 3.90
title                             2970                 2.27
description                       1656                 1.27
idsAlt                               0                 0.00
sources                              0                 0.00
activeContactsCount                  0  

### Data Prep

In [41]:
df_bigdatis_std = df_bigdatis.drop(columns=[col for col in df_bigdatis.columns if df_bigdatis[col].isna().all()])
df_bigdatis_std.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 130826 entries, 0 to 130825
Data columns (total 31 columns):
 #   Column                      Non-Null Count   Dtype  
---  ------                      --------------   -----  
 0   id                          130826 non-null  int64  
 1   idsAlt                      130826 non-null  object 
 2   sources                     130826 non-null  object 
 3   title                       127856 non-null  object 
 4   description                 129170 non-null  object 
 5   price                       104541 non-null  float64
 6   area                        68567 non-null   float64
 7   flags                       130826 non-null  object 
 8   thumbnailUrl                103208 non-null  object 
 9   images                      130826 non-null  object 
 10  imageUrls                   130826 non-null  object 
 11  locationId                  117431 non-null  float64
 12  sellerTypes                 130826 non-null  object 
 13  contacts      

In [42]:
def normalize_schema(df, mapping):
    # rename columns to master names
    df = df.rename(columns=mapping)

    # add missing columns
    for col in MASTER_SCHEMA:
        if col not in df.columns:
            df[col] = None

    # keep only master schema order
    df = df[MASTER_SCHEMA]

    return df

In [43]:
location_map = pd.read_csv("location_map.csv")
df_bigdatis_std = df_bigdatis_std.merge(location_map, on="locationId", how="left")


# only fill nulls with mapped values
for col in ["gouvernerat", "ville", "adresse", "code_postal"]:
    df_bigdatis_std[col] = df_bigdatis_std[col].combine_first(df_bigdatis_std[f"{col}"])
df_bigdatis_std


,id,idsAlt,sources,title,description,price,area,flags,thumbnailUrl,images,imageUrls,locationId,sellerTypes,contacts,activeContactsCount,adsCount,activeAdsCount,sourcesCount,activeSourcesCount,firstSeenAt,createdAt,modifiedAt,priceDroppedAt,timestamp,priceTimestamp,comments,commentsCount,properties.transactionType,properties.sellerType,properties.propertyType,properties.typology,gouvernerat,ville,adresse,code_postal
0,921274,[],"[{'sourceId': 2, 'lastModified': 1771364245, '...",Location studio manzah 9 bien équipé,Un large studio de 60 métres carrés avec cuisi...,800.00,60.00,[],56202/28101387_f3327061e9695940.jpg,[{'url': 'https://storage.googleapis.com/tayar...,[https://storage.googleapis.com/tayara-migrati...,5207.00,[private],"[{'sellerType': 'private', 'contactName': 'Moh...",1,2,1,1,1,2024-12-30 23:00:20,2024-12-30 22:53:14,2026-02-17 21:37:25,2024-12-30 22:53:14,2026-02-17 21:37:25,2024-12-30 22:53:14,[],0,rental,private,flat,s+1,Tunis,Makni,"calme, sécurisé et entouré de villas",NaN
1,1211799,[],"[{'sourceId': 5, 'lastModified': 1771364733, '...",S2 en location,Cet appartement jamais habité est situé au rez...,850.00,85.00,[new],https://www.mubawab-media.com/ad/8/269/642F/h/...,[{'url': 'https://www.mubawab-media.com/ad/8/2...,[https://www.mubawab-media.com/ad/8/269/642F/h...,4901.00,[agency],"[{'sellerType': 'agency', 'contactName': 'MFK ...",1,3,3,3,3,2025-12-09 19:15:04,2025-12-09 19:02:24,2026-02-17 21:45:33,2025-12-09 19:02:24,2026-02-17 21:45:33,2025-12-09 19:02:24,[],0,rental,agency,flat,s+2,Tunis,El Omrane Supérieur,NaN,NaN
2,1186745,[],"[{'sourceId': 5, 'lastModified': 1771361145, '...",ENNASR 2 à louer spacieux appt S3 coté collège,ENNASR 2 ; à Louer spacieux Appartement S3 dan...,NaN,170.00,[],https://www.mubawab-media.com/ad/8/237/072F/h/...,[{'url': 'https://www.mubawab-media.com/ad/8/2...,[https://www.mubawab-media.com/ad/8/237/072F/h...,300.00,[agency],"[{'sellerType': 'agency', 'contactName': 'Immo...",1,2,2,2,2,2025-10-18 08:30:36,2025-10-18 08:30:33,2026-02-17 20:45:45,2025-10-18 08:30:33,2026-02-17 20:45:45,2025-10-18 08:30:33,[],0,rental,agency,flat,s+3,NaN,NaN,NaN,NaN
3,1234370,[],"[{'sourceId': 5, 'lastModified': 1771361145, '...",Un appartement S3 vide a la Marsa,1Step2House vous propose à la location un appa...,2700.00,140.00,[],56075/28037868_c8d8d9eb61f7f3cc.jpg,[{'url': 'https://www.mubawab-media.com/ad/8/2...,[https://www.mubawab-media.com/ad/8/285/913F/h...,5220.00,[agency],"[{'sellerType': 'agency', 'contactName': '1Ste...",1,1,1,1,1,2026-01-13 20:00:21,2026-01-13 20:00:10,2026-02-17 20:45:45,2026-01-13 20:00:10,2026-02-17 20:45:45,2026-01-13 20:00:10,[],0,rental,agency,flat,s+3,Tunis,Marsa,NaN,NaN
4,1234371,[],"[{'sourceId': 5, 'lastModified': 1771361145, '...",Un appartement S4 vide à la Marsa,1Step2House vous propose à la location un appa...,4500.00,200.00,[],56075/28037869_30eac04000000000.jpg,[{'url': 'https://www.mubawab-media.com/ad/8/2...,[https://www.mubawab-media.com/ad/8/285/909F/h...,5220.00,[agency],"[{'sellerType': 'agency', 'contactName': '1Ste...",1,1,1,1,1,2026-01-13 20:00:21,2026-01-13 20:00:10,2026-02-17 20:45:45,2026-01-13 20:00:10,2026-02-17 20:45:45,2026-01-13 20:00:10,[],0,rental,agency,flat,s+4,Tunis,Marsa,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
130821,1240792,[],"[{'sourceId': 12, 'lastModified': 1769577174, ...",Vend petite villa,Vend petite villa adorable \nTitre bleu très j...,NaN,NaN,[],56121/28060950_41a6fcf964243f8f.jpg,[{'url': 'https://b3g4.fra01.idrivee2-68.com/i...,[https://b3g4.fra01.idrivee2-68.com/images-mir...,160.00,[private],[],0,1,1,1,1,1769577462,1769577174.00,1769577174.00,1769577174.00,1769577174.00,1769577174.00,[],0,sale,private,house,NaN,Tunis,Carthage,NaN,NaN
130822,1240796,[],"[{'sourceId': 1, 'lastModified': 1771369200, '...",Appt s1 à la nouvelle soukra,agence immobilière immoliv vous propose à la l...,1000.00,65.00,[new],http:/

In [44]:
df_bigdatis_std.drop(columns=["images"], inplace=True)

mapping_bigdatis= {
    "thumbnailUrl": "url",
    "properties.transactionType": "contrat",
    "description": "description",
    "title": "titre",
    "properties.typology": "pieces",
    "price": "prix",
    "properties.propertyType": "type",
    "area": "surface",
    "flags": "caracteristiques",
    "imageUrls": "images",
    "modifiedAt": "date_publication"
}
df_bigdatis_std = normalize_schema(df_bigdatis_std, mapping_bigdatis)
df_bigdatis_std.info()
df_bigdatis_std

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 130826 entries, 0 to 130825
Data columns (total 29 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   url               103208 non-null  object 
 1   contrat           125726 non-null  object 
 2   type              130826 non-null  object 
 3   prix              104541 non-null  float64
 4   titre             127856 non-null  object 
 5   description       129170 non-null  object 
 6   surface           68567 non-null   float64
 7   pieces            93384 non-null   object 
 8   etage             0 non-null       object 
 9   code_postal       308 non-null     object 
 10  adresse           36934 non-null   object 
 11  ville             102103 non-null  object 
 12  gouvernerat       93909 non-null   object 
 13  latitude          0 non-null       object 
 14  longitude         0 non-null       object 
 15  date_publication  119489 non-null  object 
 16  date_scraping     0 

,url,contrat,type,prix,titre,description,surface,pieces,etage,code_postal,adresse,ville,gouvernerat,latitude,longitude,date_publication,date_scraping,standing,annee_constr,ecole,pharmacie,hopital,marche,magasin,restaurant,bus,railway,caracteristiques,images
0,56202/28101387_f3327061e9695940.jpg,rental,flat,800.00,Location studio manzah 9 bien équipé,Un large studio de 60 métres carrés avec cuisi...,60.00,s+1,None,NaN,"calme, sécurisé et entouré de villas",Makni,Tunis,None,None,2026-02-17 21:37:25,None,None,None,None,None,None,None,None,None,None,None,[],[https://storage.googleapis.com/tayara-migrati...
1,https://www.mubawab-media.com/ad/8/269/642F/h/...,rental,flat,850.00,S2 en location,Cet appartement jamais habité est situé au rez...,85.00,s+2,None,NaN,NaN,El Omrane Supérieur,Tunis,None,None,2026-02-17 21:45:33,None,None,None,None,None,None,None,None,None,None,None,[new],[https://www.mubawab-media.com/ad/8/269/642F/h...
2,https://www.mubawab-media.com/ad/8/237/072F/h/...,rental,flat,NaN,ENNASR 2 à louer spacieux appt S3 coté collège,ENNASR 2 ; à Louer spacieux Appartement S3 dan...,170.00,s+3,None,NaN,NaN,NaN,NaN,None,None,2026-02-17 20:45:45,None,None,None,None,None,None,None,None,None,None,None,[],[https://www.mubawab-media.com/ad/8/237/072F/h...
3,56075/28037868_c8d8d9eb61f7f3cc.jpg,rental,flat,2700.00,Un appartement S3 vide a la Marsa,1Step2House vous propose à la location un appa...,140.00,s+3,None,NaN,NaN,Marsa,Tunis,None,None,2026-02-17 20:45:45,None,None,None,None,None,None,None,None,None,None,None,[],[https://www.mubawab-media.com/ad/8/285/913F/h...
4,56075/28037869_30eac04000000000.jpg,rental,flat,4500.00,Un appartement S4 vide à la Marsa,1Step2House vous propose à la location un appa...,200.00,s+4,None,NaN,NaN,Marsa,Tunis,None,None,2026-02-17 20:45:45,None,None,None,None,None,None,None,None,None,None,None,[],[https://www.mubawab-media.com/ad/8/285/909F/h...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
130821,56121/28060950_41a6fcf964243f8f.jpg,sale,house,NaN,Vend petite villa,Vend petite villa adorable \nTitre bleu très j...,NaN,NaN,None,NaN,NaN,Carthage,Tunis,None,None,1769577174.00,None,None,None,None,None,None,None,None,None,None,None,[],[https://b3g4.fra01.idrivee2-68.com/images-mir...
130822,http://tunisie-annonce.com/upload2/202512/tuni...,rental,flat,1000.00,Appt s1 à la nouvelle soukra,agence immobilière immoliv vous propose à la l...,65.00,s+1,None,NaN,Nouvelle Soukra,Soukra,Tunis,None,None,1771369200.00,None,None,None,None,None,None,None,None,None,None,None,[new],[https://b3g4.fra01.idrivee2-68.com/images-mir...
130823,56121/28060957_713139ec71a94d77.jpg,NaN,buildingLot,NaN,En #exclusivité \nL'agence ilyne immobilier #s...,En #exclusivité \nL'agence ilyne immobilier #s...,840.00,NaN,None,NaN,Lafrane,Sfax,Sfax,None,None,1769577174.00,None,None,None,None,None,None,None,None,None,None,None,[],[https://b3g4.fra01.idrivee2-68.com/images-mir...
130824,56121/28060963_d8f8e8c2d8dc90f4.jpg,sale,flat,290000.00,Avendre s3,Agence immo les jasmins \n\nA VENDRE🏠\n📣notre ...,150.00,s+3,None,NaN,NaN,Carthage,Tunis,None,None,1769577227.00,None,None,None,None,None,None,None,None,None,None,None,[],[https://b3g4.fra01.idrivee2-68.com/images-mir...


#### Data type conversion

In [45]:
df_bigdatis_std["prix"] = (
    df_bigdatis_std["prix"]
    .astype(str)
    .str.replace(r"[^\d.]", "", regex=True)  # keep only numbers
)

df_bigdatis_std["prix"] = pd.to_numeric(df_bigdatis_std["prix"], errors="coerce")

df_bigdatis_std["surface"] = pd.to_numeric(df_bigdatis_std["surface"], errors="coerce")

df_bigdatis_std["etage"] = pd.to_numeric(df_bigdatis_std["etage"], errors="coerce")
df_bigdatis_std["annee_constr"] = pd.to_numeric(df_bigdatis_std["annee_constr"], errors="coerce")

df_bigdatis_std["pieces"] = pd.to_numeric(df_bigdatis_std["pieces"], errors="coerce")

df_bigdatis_std["date_publication"] = pd.to_datetime(df_bigdatis_std["date_publication"], errors="coerce")

df_bigdatis_std


,url,contrat,type,prix,titre,description,surface,pieces,etage,code_postal,adresse,ville,gouvernerat,latitude,longitude,date_publication,date_scraping,standing,annee_constr,ecole,pharmacie,hopital,marche,magasin,restaurant,bus,railway,caracteristiques,images
0,56202/28101387_f3327061e9695940.jpg,rental,flat,800.00,Location studio manzah 9 bien équipé,Un large studio de 60 métres carrés avec cuisi...,60.00,NaN,NaN,NaN,"calme, sécurisé et entouré de villas",Makni,Tunis,None,None,2026-02-17 21:37:25,None,None,NaN,None,None,None,None,None,None,None,None,[],[https://storage.googleapis.com/tayara-migrati...
1,https://www.mubawab-media.com/ad/8/269/642F/h/...,rental,flat,850.00,S2 en location,Cet appartement jamais habité est situé au rez...,85.00,NaN,NaN,NaN,NaN,El Omrane Supérieur,Tunis,None,None,2026-02-17 21:45:33,None,None,NaN,None,None,None,None,None,None,None,None,[new],[https://www.mubawab-media.com/ad/8/269/642F/h...
2,https://www.mubawab-media.com/ad/8/237/072F/h/...,rental,flat,NaN,ENNASR 2 à louer spacieux appt S3 coté collège,ENNASR 2 ; à Louer spacieux Appartement S3 dan...,170.00,NaN,NaN,NaN,NaN,NaN,NaN,None,None,2026-02-17 20:45:45,None,None,NaN,None,None,None,None,None,None,None,None,[],[https://www.mubawab-media.com/ad/8/237/072F/h...
3,56075/28037868_c8d8d9eb61f7f3cc.jpg,rental,flat,2700.00,Un appartement S3 vide a la Marsa,1Step2House vous propose à la location un appa...,140.00,NaN,NaN,NaN,NaN,Marsa,Tunis,None,None,2026-02-17 20:45:45,None,None,NaN,None,None,None,None,None,None,None,None,[],[https://www.mubawab-media.com/ad/8/285/913F/h...
4,56075/28037869_30eac04000000000.jpg,rental,flat,4500.00,Un appartement S4 vide à la Marsa,1Step2House vous propose à la location un appa...,200.00,NaN,NaN,NaN,NaN,Marsa,Tunis,None,None,2026-02-17 20:45:45,None,None,NaN,None,None,None,None,None,None,None,None,[],[https://www.mubawab-media.com/ad/8/285/909F/h...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
130821,56121/28060950_41a6fcf964243f8f.jpg,sale,house,NaN,Vend petite villa,Vend petite villa adorable \nTitre bleu très j...,NaN,NaN,NaN,NaN,NaN,Carthage,Tunis,None,None,NaT,None,None,NaN,None,None,None,None,None,None,None,None,[],[https://b3g4.fra01.idrivee2-68.com/images-mir...
130822,http://tunisie-annonce.com/upload2/202512/tuni...,rental,flat,1000.00,Appt s1 à la nouvelle soukra,agence immobilière immoliv vous propose à la l...,65.00,NaN,NaN,NaN,Nouvelle Soukra,Soukra,Tunis,None,None,NaT,None,None,NaN,None,None,None,None,None,None,None,None,[new],[https://b3g4.fra01.idrivee2-68.com/images-mir...
130823,56121/28060957_713139ec71a94d77.jpg,NaN,buildingLot,NaN,En #exclusivité \nL'agence ilyne immobilier #s...,En #exclusivité \nL'agence ilyne immobilier #s...,840.00,NaN,NaN,NaN,Lafrane,Sfax,Sfax,None,None,NaT,None,None,NaN,None,None,None,None,None,None,None,None,[],[https://b3g4.fra01.idrivee2-68.com/images-mir...
130824,56121/28060963_d8f8e8c2d8dc90f4.jpg,sale,flat,290000.00,Avendre s3,Agence immo les jasmins \n\nA VENDRE🏠\n📣notre ...,150.00,NaN,NaN,NaN,NaN,Carthage,Tunis,None,None,NaT,None,None,NaN,None,None,None,None,None,None,None,None,[],[https://b3g4.fra01.idrivee2-68.com/images-mir...


## Tecnocasa

In [46]:
data = pd.read_json('datasets/tecnocasa/data.json')
df_tecnocasa = json_normalize(data.to_dict('records'))
df_tecnocasa.info()
df_tecnocasa

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1795 entries, 0 to 1794
Columns: 358 entries, ad_type to services.group.9.nome-micro
dtypes: bool(7), datetime64[ns](1), float64(100), int64(26), object(224)
memory usage: 4.8+ MB


,ad_type,id,address,bathrooms,bedrooms_small,country,data,description,detail_url,detail_url_qr,discount,discount_percentage,estate_other_contract,estate_parent,estates_child,estates_sibling,exclusive,exposure_primary,exposure_secondary,fund_id,height_internal,is_discounted,is_hidden,is_saved,is_tag,last_published_at,latitude,longitude,network,numeric_price,numeric_surface,pdf,publish_map,previous_price,price,price_alternative,private_negotiation,quarter,rooms,services,show_map,status_internal,subtitle,surface,tecnovalore,title,top,data_pdf,agency.aicat,agency.active,agency.address,agency.city.capital,agency.city.id,agency.city.title,agency.city.slug,agency.country,agency.district,agency.district_slug,agency.distance,agency.email,agency.id,agency.images,agency.latitude,agency.longitude,agency.mobile,agency.name,agency.network,agency.opening_hours,agency.phone,agency.phone_proxy,agency.phone_real,agency.province.id,agency.province.title,agency.province.slug,agency.pubbliweb,agency.region.id,agency.region.title,agency.region.slug,agency.sector,agency.status,agency.team,agency.url_agency,agency.url_facebook,agency.url_find,agency.url_image,agency.url_instagram,agency.url_linkedin,agency.url_sell,agency.url_short,agency.url_twitter,agency.vat,agency.whatsapp,agency.zip_code,agency.url_site,city.capital,city.id,city.title,city.slug,contract.id,contract.slug,contract.slug_seohub,contract.slug_estate,contract.title,costs.price,costs.box_price,costs.mortgage_payment,costs.mortgage_payment_value,costs.expenses,dates.build_year,district.id,district.title,district.slug,energy_data.class,energy_data.class_emissions,energy_data.certification_date,energy_data.certification_number,energy_data.certification_type,energy_data.efficiency,energy_data.emissions,energy_data.ep_nren,energy_data.ep_ren,energy_data.icone,energy_data.icone_emissions,energy_data.smile_ce,energy_data.smile_ci,energy_data.building_state,energy_data.heating,energy_data.heating_type,energy_data.heating_power,features.id,features.floor,features.floors,features.box,features.car_places,features.balconies,features.terraces,features.bedrooms,features.furnitured,features.air_conditioning,features.elevator,features.heating,features.garden,features.free,features.category,features.build_year,features.property_type,features.concierge,features.renovation_year,features.inside_renovation_year,media.floor_plans,media.has_realistico,media.has_smartvt360,media.images,media.images_ai,media.map.latitude,media.map.longitude,media.map.exact,media.video,media.virtual_tour,mortgage_data.duration,mortgage_data.external,mortgage_data.fixed_rate,mortgage_data.period,mortgage_data.rate_visible,mortgage_data.user.particulier,mortgage_data.user.professionnel,mortgage_data.user.tre,mortgage_data.payment_type_default,mortgage_data.financial_agency,mortgage_data.financial_agent,mortgage_data.mortgage_duration.0.DURATA_PRO,mortgage_data.mortgage_duration.0.TASSO__FIN,mortgage_data.mortgage_duration.0.CODICE_PRO_FIN,mortgage_data.mortgage_duration.84.DURATA_PRO,mortgage_data.mortgage_duration.84.TASSO__FIN,mortgage_data.mortgage_duration.84.CODICE_PRO_FIN,mortgage_data.mortgage_duration.120.DURATA_PRO,mortgage_data.mortgage_duration.120.TASSO__FIN,mortgage_data.mortgage_duration.120.CODICE_PRO_FIN,mortgage_data.mortgage_duration.180.DURATA_PRO,mortgage_data.mortgage_duration.180.TASSO__FIN,mortgage_data.mortgage_duration.180.CODICE_PRO_FIN,mortgage_data.mortgage_duration.240.DURATA_PRO,mortgage_data.mortgage_duration.240.TASSO__FIN,mortgage_data.mortgage_duration.240.CODICE_PRO_FIN,mortgage_data.mortgage_duration.300.DURATA_PRO,mortgage_data.mortgage_duration.300.TASSO__FIN,mortgage_data.mortgage_duration.300.CODICE_PRO_FIN,mortgage_data.url,points_of_interest.school,points_of_interest.pharmacy,points_of_interest.hospital,points_of_interest.market,points_of_interest.shop,points_of_interest.bar,points_of_interest.restaurant,province.id,province.title,province.slug,region.title,region.slug,sector.id,sector.sl

In [47]:
df_tecnocasa = df_tecnocasa.replace(
    ["", " ", "NA", "N/A", "na", "null", "None", None],
    np.nan
)
null_table = (
    pd.DataFrame({
        "Null Count": df_tecnocasa.isna().sum(),
        "Null Percentage (%)": df_tecnocasa.isna().mean() * 100
    })
    .round(2)
    .sort_values(by="Null Percentage (%)", ascending=False)
)

print(null_table)

                                    Null Count  Null Percentage (%)
services.group.17.surface                 1795               100.00
services.group.13.nome-descrizione        1795               100.00
services.group.13.surface                 1795               100.00
services.group.14.nome-descrizione        1795               100.00
quarter                                   1795               100.00
...                                        ...                  ...
publish_map                                  0                 0.00
price                                        0                 0.00
address                                      0                 0.00
id                                           0                 0.00
ad_type                                      0                 0.00

[358 rows x 2 columns]


### Data Prep

In [48]:
df_tecnocasa_std = df_tecnocasa.drop(columns=[col for col in df_tecnocasa.columns if df_tecnocasa[col].isna().all()])
df_tecnocasa_std.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1795 entries, 0 to 1794
Columns: 270 entries, ad_type to services.group.9.nome-micro
dtypes: bool(7), datetime64[ns](1), float64(69), int64(26), object(167)
memory usage: 3.6+ MB


**Combine caracteristics into one column**

In [49]:
columns = [
    "services.group.2.nome-macro",
    "services.group.0.nome-macro",
    "services.group.6.nome-macro",
    "services.group.1.nome-macro",
    "services.group.4.nome-macro",
    "services.group.5.nome-macro",
    "services.group.7.nome-macro",
    "services.group.8.nome-macro",
    "services.group.9.nome-macro",
    "services.group.10.nome-macro",
    "services.group.11.nome-macro",
]

# Combine all 3 columns into one list per row
def combine_services(row):
    values = []
    for col in columns:
        if col in df_tecnocasa_std.columns and pd.notna(row[col]):
            values.extend(str(row[col]).split(","))
    return [v.strip() for v in values if v.strip() != ""]

df_tecnocasa_std["caracteristiques"] = df_tecnocasa_std.apply(combine_services, axis=1)
df_tecnocasa_std["caracteristiques"].value_counts()

caracteristiques
[]                                                                                                                 1450
[Balcon]                                                                                                             62
[Terrasse]                                                                                                           31
[Ascenseur]                                                                                                          11
[Jardin]                                                                                                              9
                                                                                                                   ... 
[Ascenseur, Alarme, Balcon, Vidéophone, Gardien, Système anti-incendie, Allée (voie d'accès), Place de parking]       1
[Gardien, Ascenseur, Vidéophone]                                                                                      1
[cour intérieure, Balco

In [50]:
def add_features_to_caracteristiques(row):
    # Ensure caracteristiques is a list
    if isinstance(row['caracteristiques'], str):
        try:
            features_list = ast.literal_eval(row['caracteristiques'])
        except:
            features_list = []
    elif isinstance(row['caracteristiques'], list):
        features_list = row['caracteristiques']
    else:
        features_list = []

    if row.get('afeatures.air_conditioning') not in [np.nan, "none", "na", "null", "None",'non',"non clim","no clim","non climatisé","no"] :
        features_list.append("clim")

    if row.get('features.elevator') not in [np.nan, "none", "na", "null", "None","non","no","No","Non"]:
        features_list.append("ascenseur")

    if row.get('features.heating') not in [np.nan, "none", "na", "null", "None","non","no","No","Non"]:
        features_list.append("chauffage")

    if row.get('features.garden') not in [np.nan, "none", "na", "null", "None","non","no","No","Non"]:
        features_list.append("jardin")

    return list(set(features_list))  # remove duplicates if any

df_tecnocasa_std['caracteristiques'] = df_tecnocasa_std.apply(add_features_to_caracteristiques, axis=1)

df_tecnocasa_std['caracteristiques'].value_counts()

caracteristiques
[clim]                                                                                                                                         1224
[clim, chauffage]                                                                                                                               184
[chauffage, clim, Balcon]                                                                                                                        51
[ascenseur, clim]                                                                                                                                23
[Terrasse, clim, chauffage]                                                                                                                      22
                                                                                                                                               ... 
[ascenseur, Potager/Jardin Potager, Garage, clim]                                              

In [51]:
def nearest_distance_by_class(val, target_class):
    """Extract nearest distance for a specific class."""
    if val is None:
        return None
    if isinstance(val, str):
        try:
            val = ast.literal_eval(val)
        except:
            return None
    if not isinstance(val, list) or len(val) == 0:
        return None
    
    distances = []
    for item in val:
        if not isinstance(item, dict):
            continue
        if item.get("class", "").lower() != target_class.lower():
            continue
        dist = str(item.get("distance", "")).strip().replace(",", ".")
        nums = re.findall(r"[\d.]+", dist)
        if not nums:
            continue
        num = float(nums[0])
        distances.append(num * 1000 if "km" in dist.lower() else num)
    
    return min(distances) if distances else None

df_tecnocasa_std["railway"] = df_tecnocasa_std["points_of_interest.public_transport"].apply(lambda x: nearest_distance_by_class(x, "railway"))
df_tecnocasa_std["bus"]     = df_tecnocasa_std["points_of_interest.public_transport"].apply(lambda x: nearest_distance_by_class(x, "bus"))

In [52]:
def normalize_schema(df, mapping):
    # rename columns to master names
    df = df.rename(columns=mapping)
    
    # add missing columns
    for col in MASTER_SCHEMA:
        if col not in df.columns:
            df[col] = None

    # keep only master schema order
    df = df[MASTER_SCHEMA]

    return df

In [53]:
df_tecnocasa_std.drop(columns=["surface"], inplace=True)

mapping_tecnocasa= {
    "detail_url": "url",
    "transaction": "contrat",
    "description": "description",
    "title": "titre",
    "rooms": "pieces",
    "numeric_price": "prix",
    "latitude": "latitude",
    "province.slug":"gouvernerat",
    "longitude": "longitude",
    "district.slug": "ville",
    "address": "adresse",
    "type.slug": "type",
    "features.floor": "etage",
    "features.category": "standing",
    "features.build_year": "annee_constr",
    "numeric_surface": "surface",
    "last_published_at": "date_publication",
    "contract.title": "contrat",
    "points_of_interest.school": "ecole",
    "points_of_interest.pharmacy": "pharmacie",
    "points_of_interest.hospital": "hopital",
    "points_of_interest.market": "marche",
    "points_of_interest.shop": "magasin",
    "points_of_interest.bar": "bar",
    "points_of_interest.restaurant": "restaurant",
    "media.images": "images"
}
df_tecnocasa_std = normalize_schema(df_tecnocasa_std, mapping_tecnocasa)
df_tecnocasa_std.info()
df_tecnocasa_std

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1795 entries, 0 to 1794
Data columns (total 29 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   url               1795 non-null   object        
 1   contrat           1795 non-null   object        
 2   type              1795 non-null   object        
 3   prix              1795 non-null   int64         
 4   titre             1795 non-null   object        
 5   description       1795 non-null   object        
 6   surface           1795 non-null   int64         
 7   pieces            1079 non-null   object        
 8   etage             1288 non-null   object        
 9   code_postal       0 non-null      object        
 10  adresse           1795 non-null   object        
 11  ville             1795 non-null   object        
 12  gouvernerat       1795 non-null   object        
 13  latitude          1795 non-null   float64       
 14  longitude         1795 n

,url,contrat,type,prix,titre,description,surface,pieces,etage,code_postal,adresse,ville,gouvernerat,latitude,longitude,date_publication,date_scraping,standing,annee_constr,ecole,pharmacie,hopital,marche,magasin,restaurant,bus,railway,caracteristiques,images
0,https://www.tecnocasa.tn/vendre/villa/bizerte/...,Vente,villa,550000,Villa jumelée en vente,<p>Villa à deux étages offrant de beaux volume...,196,NaN,2,None,Rue Bhira,bizerte,bizerte,37.29,9.85,2026-01-21 12:40:58,None,Moyen standing,NaN,"[{'name': 'Lycée El Bhira', 'class': 'school',...","[{'name': 'Pharmacie de nuit', 'class': 'pharm...",[{'name': 'Unité De Diagnostic Anténatal UDANB...,"[{'name': 'El Corniche', 'class': 'grocery', '...","[{'name': 'Proxi', 'class': 'shop', 'subclass'...","[{'name': 'Bake & Bake', 'class': 'restaurant'...",NaN,NaN,[clim],"[{'id': 353558, 'order': 1, 'url': {'card': 'h..."
1,https://www.tecnocasa.tn/vendre/appartement/bi...,Vente,appartement,330000,S+2 en vente,"<p> Cet appartement <strong data-start=""125"" d...",130,3 pièces,5,None,Rue soltana,bizerte,bizerte,37.29,9.85,2026-01-13 15:09:03,None,Haut standing,2016.00,"[{'name': 'Lycée El Bhira', 'class': 'school',...","[{'name': 'Pharmacie de nuit', 'class': 'pharm...",[{'name': 'Unité De Diagnostic Anténatal UDANB...,"[{'name': 'Carrefour Market', 'class': 'grocer...","[{'name': 'Frip', 'class': 'shop', 'subclass':...","[{'name': 'Bake & Bake', 'class': 'restaurant'...",3000.00,NaN,[clim],"[{'id': 357122, 'order': 1, 'url': {'card': 'h..."
2,https://www.tecnocasa.tn/vendre/appartement/bi...,Vente,appartement,200000,Triplex en vente,"<p data-start=""0"" data-end=""709"" data-is-last-...",205,NaN,3,None,Borj taleb,bizerte,bizerte,37.28,9.84,2025-12-15 18:06:31,None,Haut standing,NaN,"[{'name': 'Écoles', 'class': 'school', 'subcla...","[{'name': 'Pharmacie de nuit', 'class': 'pharm...",[{'name': 'Unité De Diagnostic Anténatal UDANB...,"[{'name': 'Carrefour Market', 'class': 'grocer...","[{'name': 'Frip', 'class': 'shop', 'subclass':...","[{'name': 'The Garage', 'class': 'restaurant',...",2400.00,2600.00,[clim],"[{'id': 351599, 'order': 1, 'url': {'card': 'h..."
3,https://www.tecnocasa.tn/vendre/appartement/bi...,Vente,appartement,150000,S+2 en vente,<p>Cet appartemen Situé au <strong>deuxième ét...,51,3 pièces,1,None,Beb mateur,bizerte,bizerte,37.27,9.87,2025-09-03 11:57:17,None,Moyen standing,NaN,"[{'name': 'Ecole Jean Giono', 'class': 'school...","[{'name': 'Pharmacie Feu Hamadi Terrasse', 'cl...","[{'name': 'Hôpital Militaire de Bizerte', 'cla...","[{'name': 'Société Carthage Distribution "" SCD...","[{'name': 'Proxi', 'class': 'shop', 'subclass'...","[{'name': 'Restaurants', 'class': 'restaurant'...",430.00,90.00,[clim],"[{'id': 317858, 'order': 1, 'url': {'card': 'h..."
4,https://www.tecnocasa.tn/vendre/appartement/bi...,Vente,appartement,330000,S+2 en vente,"<p> cet appartement <strong data-start=""82"" da...",100,3 pièces,NaN,None,Rue Rawebi,bizerte,bizerte,37.28,9.87,2025-12-04 18:17:08,None,Haut standing,2025.00,"[{'name': 'ạlmdrsẗ ạlạ̹btdạỷyẗ bḥy ạlhnạʾ', 'c...","[{'name': 'Pharmacie De nuit', 'class': 'pharm...",[{'name': 'Unité De Diagnostic Anténatal UDANB...,"[{'name': 'El Corniche', 'class': 'grocery', '...","[{'name': 'Proxi', 'class': 'shop', 'subclass'...","[{'name': 'Bake & Bake', 'class': 'restaurant'...",1600.00,2000.00,[clim],"[{'id': 348778, 'order': 1, 'url': {'card': 'h..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1790,https://www.tecnocasa.tn/louer/appartement/sou...,Location,appartement,1800,S+3 en location,"<p data-start=""76"" data-end=""390"">Cet appartem...",160,4 pièces,2 (étage moyen),None,Route de la Plage,khezema-est,sousse,35.87,10.61,2025-08-16 15:12:55,None,Moyen standing,2012.00,"[{'name': 'El Bhaier', 'class': 'school', 'sub...","[{'name': 'Pharmacie', 'class': 'pharmacy', 's...","[{'name': 'Clinique les Oliviers', 'class': 'h...","[{'name': 'Monoprix', 'class': 'grocery'

In [54]:
poi_columns = [
    "ecole",
    "pharmacie",
    "hopital",
    "marche",
    "magasin",
    "restaurant",
]

In [55]:
def nearest_distance_meters(val):
    """Extract the nearest POI distance in meters."""
    if val is None:
        return None
    if isinstance(val, str):
        try:
            val = ast.literal_eval(val)
        except:
            return None
    if not isinstance(val, list) or len(val) == 0:
        return None
    
    distances = []
    for item in val:
        if not isinstance(item, dict):
            continue
        dist = str(item.get("distance", "")).strip().replace(",", ".")
        if not dist:
            continue
        nums = re.findall(r"[\d.]+", dist)
        if not nums:
            continue
        num = float(nums[0])
        if "km" in dist.lower():
            distances.append(num * 1000)
        else:
            distances.append(num)
    
    return min(distances) if distances else None


for col in poi_columns:
    df_tecnocasa_std[f"{col}_nearest"] = df_tecnocasa_std[col].apply(nearest_distance_meters)

In [56]:
df_tecnocasa_std

,url,contrat,type,prix,titre,description,surface,pieces,etage,code_postal,adresse,ville,gouvernerat,latitude,longitude,date_publication,date_scraping,standing,annee_constr,ecole,pharmacie,hopital,marche,magasin,restaurant,bus,railway,caracteristiques,images,ecole_nearest,pharmacie_nearest,hopital_nearest,marche_nearest,magasin_nearest,restaurant_nearest
0,https://www.tecnocasa.tn/vendre/villa/bizerte/...,Vente,villa,550000,Villa jumelée en vente,<p>Villa à deux étages offrant de beaux volume...,196,NaN,2,None,Rue Bhira,bizerte,bizerte,37.29,9.85,2026-01-21 12:40:58,None,Moyen standing,NaN,"[{'name': 'Lycée El Bhira', 'class': 'school',...","[{'name': 'Pharmacie de nuit', 'class': 'pharm...",[{'name': 'Unité De Diagnostic Anténatal UDANB...,"[{'name': 'El Corniche', 'class': 'grocery', '...","[{'name': 'Proxi', 'class': 'shop', 'subclass'...","[{'name': 'Bake & Bake', 'class': 'restaurant'...",NaN,NaN,[clim],"[{'id': 353558, 'order': 1, 'url': {'card': 'h...",350.00,1800.00,2300.00,1200.00,2900.00,1500.00
1,https://www.tecnocasa.tn/vendre/appartement/bi...,Vente,appartement,330000,S+2 en vente,"<p> Cet appartement <strong data-start=""125"" d...",130,3 pièces,5,None,Rue soltana,bizerte,bizerte,37.29,9.85,2026-01-13 15:09:03,None,Haut standing,2016.00,"[{'name': 'Lycée El Bhira', 'class': 'school',...","[{'name': 'Pharmacie de nuit', 'class': 'pharm...",[{'name': 'Unité De Diagnostic Anténatal UDANB...,"[{'name': 'Carrefour Market', 'class': 'grocer...","[{'name': 'Frip', 'class': 'shop', 'subclass':...","[{'name': 'Bake & Bake', 'class': 'restaurant'...",3000.00,NaN,[clim],"[{'id': 357122, 'order': 1, 'url': {'card': 'h...",490.00,1500.00,2100.00,1200.00,2700.00,1900.00
2,https://www.tecnocasa.tn/vendre/appartement/bi...,Vente,appartement,200000,Triplex en vente,"<p data-start=""0"" data-end=""709"" data-is-last-...",205,NaN,3,None,Borj taleb,bizerte,bizerte,37.28,9.84,2025-12-15 18:06:31,None,Haut standing,NaN,"[{'name': 'Écoles', 'class': 'school', 'subcla...","[{'name': 'Pharmacie de nuit', 'class': 'pharm...",[{'name': 'Unité De Diagnostic Anténatal UDANB...,"[{'name': 'Carrefour Market', 'class': 'grocer...","[{'name': 'Frip', 'class': 'shop', 'subclass':...","[{'name': 'The Garage', 'class': 'restaurant',...",2400.00,2600.00,[clim],"[{'id': 351599, 'order': 1, 'url': {'card': 'h...",410.00,1000.00,1800.00,1100.00,2000.00,2100.00
3,https://www.tecnocasa.tn/vendre/appartement/bi...,Vente,appartement,150000,S+2 en vente,<p>Cet appartemen Situé au <strong>deuxième ét...,51,3 pièces,1,None,Beb mateur,bizerte,bizerte,37.27,9.87,2025-09-03 11:57:17,None,Moyen standing,NaN,"[{'name': 'Ecole Jean Giono', 'class': 'school...","[{'name': 'Pharmacie Feu Hamadi Terrasse', 'cl...","[{'name': 'Hôpital Militaire de Bizerte', 'cla...","[{'name': 'Société Carthage Distribution "" SCD...","[{'name': 'Proxi', 'class': 'shop', 'subclass'...","[{'name': 'Restaurants', 'class': 'restaurant'...",430.00,90.00,[clim],"[{'id': 317858, 'order': 1, 'url': {'card': 'h...",100.00,420.00,660.00,360.00,230.00,130.00
4,https://www.tecnocasa.tn/vendre/appartement/bi...,Vente,appartement,330000,S+2 en vente,"<p> cet appartement <strong data-start=""82"" da...",100,3 pièces,NaN,None,Rue Rawebi,bizerte,bizerte,37.28,9.87,2025-12-04 18:17:08,None,Haut standing,2025.00,"[{'name': 'ạlmdrsẗ ạlạ̹btdạỷyẗ bḥy ạlhnạʾ', 'c...","[{'name': 'Pharmacie De nuit', 'class': 'pharm...",[{'name': 'Unité De Diagnostic Anténatal UDANB...,"[{'name': 'El Corniche', 'class': 'grocery', '...","[{'name': 'Proxi', 'class': 'shop', 'subclass'...","[{'name': 'Bake & Bake', 'class': 'restaurant'...",1600.00,2000.00,[clim],"[{'id': 348778, 'order': 1, 'url': {'card': 'h...",420.00,1300.00,1300.00,1200.00,1800.00,400.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1790,https://www.tecnocasa.tn/louer/appartement/sou...,Location,appartement,1800,S+3 en location,"<p data-start=""76"" data-end=""

In [57]:
df_tecnocasa_std

,url,contrat,type,prix,titre,description,surface,pieces,etage,code_postal,adresse,ville,gouvernerat,latitude,longitude,date_publication,date_scraping,standing,annee_constr,ecole,pharmacie,hopital,marche,magasin,restaurant,bus,railway,caracteristiques,images,ecole_nearest,pharmacie_nearest,hopital_nearest,marche_nearest,magasin_nearest,restaurant_nearest
0,https://www.tecnocasa.tn/vendre/villa/bizerte/...,Vente,villa,550000,Villa jumelée en vente,<p>Villa à deux étages offrant de beaux volume...,196,NaN,2,None,Rue Bhira,bizerte,bizerte,37.29,9.85,2026-01-21 12:40:58,None,Moyen standing,NaN,"[{'name': 'Lycée El Bhira', 'class': 'school',...","[{'name': 'Pharmacie de nuit', 'class': 'pharm...",[{'name': 'Unité De Diagnostic Anténatal UDANB...,"[{'name': 'El Corniche', 'class': 'grocery', '...","[{'name': 'Proxi', 'class': 'shop', 'subclass'...","[{'name': 'Bake & Bake', 'class': 'restaurant'...",NaN,NaN,[clim],"[{'id': 353558, 'order': 1, 'url': {'card': 'h...",350.00,1800.00,2300.00,1200.00,2900.00,1500.00
1,https://www.tecnocasa.tn/vendre/appartement/bi...,Vente,appartement,330000,S+2 en vente,"<p> Cet appartement <strong data-start=""125"" d...",130,3 pièces,5,None,Rue soltana,bizerte,bizerte,37.29,9.85,2026-01-13 15:09:03,None,Haut standing,2016.00,"[{'name': 'Lycée El Bhira', 'class': 'school',...","[{'name': 'Pharmacie de nuit', 'class': 'pharm...",[{'name': 'Unité De Diagnostic Anténatal UDANB...,"[{'name': 'Carrefour Market', 'class': 'grocer...","[{'name': 'Frip', 'class': 'shop', 'subclass':...","[{'name': 'Bake & Bake', 'class': 'restaurant'...",3000.00,NaN,[clim],"[{'id': 357122, 'order': 1, 'url': {'card': 'h...",490.00,1500.00,2100.00,1200.00,2700.00,1900.00
2,https://www.tecnocasa.tn/vendre/appartement/bi...,Vente,appartement,200000,Triplex en vente,"<p data-start=""0"" data-end=""709"" data-is-last-...",205,NaN,3,None,Borj taleb,bizerte,bizerte,37.28,9.84,2025-12-15 18:06:31,None,Haut standing,NaN,"[{'name': 'Écoles', 'class': 'school', 'subcla...","[{'name': 'Pharmacie de nuit', 'class': 'pharm...",[{'name': 'Unité De Diagnostic Anténatal UDANB...,"[{'name': 'Carrefour Market', 'class': 'grocer...","[{'name': 'Frip', 'class': 'shop', 'subclass':...","[{'name': 'The Garage', 'class': 'restaurant',...",2400.00,2600.00,[clim],"[{'id': 351599, 'order': 1, 'url': {'card': 'h...",410.00,1000.00,1800.00,1100.00,2000.00,2100.00
3,https://www.tecnocasa.tn/vendre/appartement/bi...,Vente,appartement,150000,S+2 en vente,<p>Cet appartemen Situé au <strong>deuxième ét...,51,3 pièces,1,None,Beb mateur,bizerte,bizerte,37.27,9.87,2025-09-03 11:57:17,None,Moyen standing,NaN,"[{'name': 'Ecole Jean Giono', 'class': 'school...","[{'name': 'Pharmacie Feu Hamadi Terrasse', 'cl...","[{'name': 'Hôpital Militaire de Bizerte', 'cla...","[{'name': 'Société Carthage Distribution "" SCD...","[{'name': 'Proxi', 'class': 'shop', 'subclass'...","[{'name': 'Restaurants', 'class': 'restaurant'...",430.00,90.00,[clim],"[{'id': 317858, 'order': 1, 'url': {'card': 'h...",100.00,420.00,660.00,360.00,230.00,130.00
4,https://www.tecnocasa.tn/vendre/appartement/bi...,Vente,appartement,330000,S+2 en vente,"<p> cet appartement <strong data-start=""82"" da...",100,3 pièces,NaN,None,Rue Rawebi,bizerte,bizerte,37.28,9.87,2025-12-04 18:17:08,None,Haut standing,2025.00,"[{'name': 'ạlmdrsẗ ạlạ̹btdạỷyẗ bḥy ạlhnạʾ', 'c...","[{'name': 'Pharmacie De nuit', 'class': 'pharm...",[{'name': 'Unité De Diagnostic Anténatal UDANB...,"[{'name': 'El Corniche', 'class': 'grocery', '...","[{'name': 'Proxi', 'class': 'shop', 'subclass'...","[{'name': 'Bake & Bake', 'class': 'restaurant'...",1600.00,2000.00,[clim],"[{'id': 348778, 'order': 1, 'url': {'card': 'h...",420.00,1300.00,1300.00,1200.00,1800.00,400.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1790,https://www.tecnocasa.tn/louer/appartement/sou...,Location,appartement,1800,S+3 en location,"<p data-start=""76"" data-end=""

In [58]:
print(df_tecnocasa_std['pieces'].value_counts())
if 'pieces' in df_tecnocasa_std.columns:
    df_tecnocasa_std['pieces'] = df_tecnocasa_std['pieces'].astype(str).str.extract('(\d+)').astype('Int64')
print(df_tecnocasa_std['pieces'].unique())

if 'etage' in df_tecnocasa_std.columns:
    manual_mapping = {
    'RDC': 0,
    'Terre': 0,
    'Sous-sol': -1,
    'Sot': -1,
    'Bas': 1,
    'Moyen': 2,
    'Elevé': 3,
    'Dernier': np.nan,  # optional: choose max floor or leave NaN
    'Grenier': np.nan   # optional: choose a value
}

def process_floor(val):
    val_str = str(val).strip()
    
    # If empty, return NaN
    if val_str == '':
        return np.nan
    
    # Try extracting first number
    num = pd.Series([int(x) for x in val_str if x.isdigit()])
    if not num.empty:
        return int(num.iloc[0])
    
    # If no number, check mapping
    return manual_mapping.get(val_str, np.nan)  # default NaN if unknown

# Apply function
df_tecnocasa_std['etage'] = df_tecnocasa_std['etage'].apply(process_floor).astype('Int64')
print(df_tecnocasa_std['etage'].unique())

pieces
3 pièces    416
2 pièces    274
4 pièces    254
1 pièce      58
5 pièces     52
6 pièces     17
7 pièces      6
8 pièces      2
Name: count, dtype: int64
<IntegerArray>
[<NA>, 3, 4, 1, 6, 2, 5, 8, 7]
Length: 9, dtype: Int64
<IntegerArray>
[2, 5, 3, 1, <NA>, 0, 4, 6, 7, 9, -1, 8]
Length: 12, dtype: Int64


#### Data type conversion

In [59]:
df_tecnocasa_std["prix"] = (
    df_tecnocasa_std["prix"]
    .astype(str)
    .str.replace(r"[^\d.]", "", regex=True)  # keep only numbers
)

df_tecnocasa_std["prix"] = pd.to_numeric(df_tecnocasa_std["prix"], errors="coerce")

df_tecnocasa_std["surface"] = pd.to_numeric(df_tecnocasa_std["surface"], errors="coerce")

df_tecnocasa_std["etage"] = pd.to_numeric(df_tecnocasa_std["etage"], errors="coerce")
df_tecnocasa_std["annee_constr"] = pd.to_numeric(df_tecnocasa_std["annee_constr"], errors="coerce")



df_tecnocasa_std["date_publication"] = pd.to_datetime(df_tecnocasa_std["date_publication"], errors="coerce")

df_tecnocasa_std


,url,contrat,type,prix,titre,description,surface,pieces,etage,code_postal,adresse,ville,gouvernerat,latitude,longitude,date_publication,date_scraping,standing,annee_constr,ecole,pharmacie,hopital,marche,magasin,restaurant,bus,railway,caracteristiques,images,ecole_nearest,pharmacie_nearest,hopital_nearest,marche_nearest,magasin_nearest,restaurant_nearest
0,https://www.tecnocasa.tn/vendre/villa/bizerte/...,Vente,villa,550000,Villa jumelée en vente,<p>Villa à deux étages offrant de beaux volume...,196,<NA>,2,None,Rue Bhira,bizerte,bizerte,37.29,9.85,2026-01-21 12:40:58,None,Moyen standing,NaN,"[{'name': 'Lycée El Bhira', 'class': 'school',...","[{'name': 'Pharmacie de nuit', 'class': 'pharm...",[{'name': 'Unité De Diagnostic Anténatal UDANB...,"[{'name': 'El Corniche', 'class': 'grocery', '...","[{'name': 'Proxi', 'class': 'shop', 'subclass'...","[{'name': 'Bake & Bake', 'class': 'restaurant'...",NaN,NaN,[clim],"[{'id': 353558, 'order': 1, 'url': {'card': 'h...",350.00,1800.00,2300.00,1200.00,2900.00,1500.00
1,https://www.tecnocasa.tn/vendre/appartement/bi...,Vente,appartement,330000,S+2 en vente,"<p> Cet appartement <strong data-start=""125"" d...",130,3,5,None,Rue soltana,bizerte,bizerte,37.29,9.85,2026-01-13 15:09:03,None,Haut standing,2016.00,"[{'name': 'Lycée El Bhira', 'class': 'school',...","[{'name': 'Pharmacie de nuit', 'class': 'pharm...",[{'name': 'Unité De Diagnostic Anténatal UDANB...,"[{'name': 'Carrefour Market', 'class': 'grocer...","[{'name': 'Frip', 'class': 'shop', 'subclass':...","[{'name': 'Bake & Bake', 'class': 'restaurant'...",3000.00,NaN,[clim],"[{'id': 357122, 'order': 1, 'url': {'card': 'h...",490.00,1500.00,2100.00,1200.00,2700.00,1900.00
2,https://www.tecnocasa.tn/vendre/appartement/bi...,Vente,appartement,200000,Triplex en vente,"<p data-start=""0"" data-end=""709"" data-is-last-...",205,<NA>,3,None,Borj taleb,bizerte,bizerte,37.28,9.84,2025-12-15 18:06:31,None,Haut standing,NaN,"[{'name': 'Écoles', 'class': 'school', 'subcla...","[{'name': 'Pharmacie de nuit', 'class': 'pharm...",[{'name': 'Unité De Diagnostic Anténatal UDANB...,"[{'name': 'Carrefour Market', 'class': 'grocer...","[{'name': 'Frip', 'class': 'shop', 'subclass':...","[{'name': 'The Garage', 'class': 'restaurant',...",2400.00,2600.00,[clim],"[{'id': 351599, 'order': 1, 'url': {'card': 'h...",410.00,1000.00,1800.00,1100.00,2000.00,2100.00
3,https://www.tecnocasa.tn/vendre/appartement/bi...,Vente,appartement,150000,S+2 en vente,<p>Cet appartemen Situé au <strong>deuxième ét...,51,3,1,None,Beb mateur,bizerte,bizerte,37.27,9.87,2025-09-03 11:57:17,None,Moyen standing,NaN,"[{'name': 'Ecole Jean Giono', 'class': 'school...","[{'name': 'Pharmacie Feu Hamadi Terrasse', 'cl...","[{'name': 'Hôpital Militaire de Bizerte', 'cla...","[{'name': 'Société Carthage Distribution "" SCD...","[{'name': 'Proxi', 'class': 'shop', 'subclass'...","[{'name': 'Restaurants', 'class': 'restaurant'...",430.00,90.00,[clim],"[{'id': 317858, 'order': 1, 'url': {'card': 'h...",100.00,420.00,660.00,360.00,230.00,130.00
4,https://www.tecnocasa.tn/vendre/appartement/bi...,Vente,appartement,330000,S+2 en vente,"<p> cet appartement <strong data-start=""82"" da...",100,3,<NA>,None,Rue Rawebi,bizerte,bizerte,37.28,9.87,2025-12-04 18:17:08,None,Haut standing,2025.00,"[{'name': 'ạlmdrsẗ ạlạ̹btdạỷyẗ bḥy ạlhnạʾ', 'c...","[{'name': 'Pharmacie De nuit', 'class': 'pharm...",[{'name': 'Unité De Diagnostic Anténatal UDANB...,"[{'name': 'El Corniche', 'class': 'grocery', '...","[{'name': 'Proxi', 'class': 'shop', 'subclass'...","[{'name': 'Bake & Bake', 'class': 'restaurant'...",1600.00,2000.00,[clim],"[{'id': 348778, 'order': 1, 'url': {'card': 'h...",420.00,1300.00,1300.00,1200.00,1800.00,400.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1790,https://www.tecnocasa.tn/louer/appartement/sou...,Location,appartement,1800,S+3 en location,"<p data-start=""76"" data-end=""390"">Cet appartem

In [60]:
def parse_image_urls(val):
    """Extract all 'detail' URLs from a list of image dicts."""
    if val is None:
        return []
    if isinstance(val, str):
        try:
            val = ast.literal_eval(val)
        except:
            return []
    if not isinstance(val, list):
        return []
    
    return [item["url"]["gallery"] for item in val if isinstance(item, dict) and "url" in item]

df_tecnocasa_std["images"] = df_tecnocasa_std["images"].apply(parse_image_urls)

In [61]:
df_tecnocasa_std["images"][0]

['https://cdn-media.medialabtc.it/tn/agencies/bzco1/estates/61927/images/353558/gallery.jpeg',
 'https://cdn-media.medialabtc.it/tn/agencies/bzco1/estates/61927/images/353557/gallery.jpeg',
 'https://cdn-media.medialabtc.it/tn/agencies/bzco1/estates/61927/images/353542/gallery.jpeg',
 'https://cdn-media.medialabtc.it/tn/agencies/bzco1/estates/61927/images/353538/gallery.jpeg',
 'https://cdn-media.medialabtc.it/tn/agencies/bzco1/estates/61927/images/353540/gallery.jpeg',
 'https://cdn-media.medialabtc.it/tn/agencies/bzco1/estates/61927/images/353541/gallery.jpeg',
 'https://cdn-media.medialabtc.it/tn/agencies/bzco1/estates/61927/images/353532/gallery.jpeg',
 'https://cdn-media.medialabtc.it/tn/agencies/bzco1/estates/61927/images/353531/gallery.jpeg',
 'https://cdn-media.medialabtc.it/tn/agencies/bzco1/estates/61927/images/353534/gallery.jpeg',
 'https://cdn-media.medialabtc.it/tn/agencies/bzco1/estates/61927/images/353535/gallery.jpeg',
 'https://cdn-media.medialabtc.it/tn/agencies/bzco

In [83]:
df_tunisie_annonce_std["contrat"].value_counts()

contrat
Location               5992
Vente                  4681
Terrain                3392
Bureaux & Commerces     872
Location vacances       206
Achat                    17
Partage                  12
Name: count, dtype: int64

## Merging datasets 

In [104]:
dataset_full_raw = pd.concat([df_monbien_std, df_tunisie_annonce_std, 
df_annonces_immo_std, df_immobilier_tn_std, df_tecnocasa_std, df_bigdatis_std], ignore_index=True)
dataset_full_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 151747 entries, 0 to 151746
Data columns (total 35 columns):
 #   Column              Non-Null Count   Dtype         
---  ------              --------------   -----         
 0   url                 124129 non-null  object        
 1   contrat             146645 non-null  object        
 2   type                151549 non-null  object        
 3   prix                125310 non-null  float64       
 4   titre               148777 non-null  object        
 5   description         150091 non-null  object        
 6   surface             86802 non-null   float64       
 7   pieces              9256 non-null    Float64       
 8   etage               1283 non-null    Int64         
 9   code_postal         308 non-null     object        
 10  adresse             54705 non-null   object        
 11  ville               119782 non-null  object        
 12  gouvernerat         110876 non-null  object        
 13  latitude            1795 non-

In [105]:
dataset_full_raw["contrat"].isna().sum()

np.int64(5102)

In [106]:
dataset_full_raw["contrat"].value_counts()

contrat
rental                 102353
sale                    19169
Location                 8549
Vente                    7782
shortTermRental          4204
Terrain                  3392
Bureaux & Commerces       872
Location vacances         206
location                   89
Achat                      17
Partage                    12
Name: count, dtype: int64

**Unify the categories in categorical data**

In [107]:
dataset_full_raw["type"].unique()

array(['Terrain', nan, 'Villa', 'Duplex', 'Appartement', 'Immeuble',
       'Bureau', 'Local commercial', 'Triplex', 'App.', 'Surfaces',
       'Maisons', 'Autre', 'Gérance', 'Fond', 'Atelier', None, 'Entrepôt',
       'villa', 'appartement', 'terrain', 'bureau', 'ferme', 'flat',
       'duplex', 'office', 'house', 'officeBuilding', 'villaFloor',
       'commercialPremise', 'buildingLot', 'warehouse', 'room',
       'farmland', 'industrialPremise', 'residentialBuilding'],
      dtype=object)

In [108]:
TYPE_MAPPING = {
    # ---------------- Apartments
    "appartement": "appartement",
    "app.": "appartement",
    "flat": "appartement",
    "villaFloor": "appartement",

    # ---------------- Houses / Villas
    "villa": "maison",
    "house": "maison",
    "maisons": "maison",
    "duplex": "maison",
    "triplex": "maison",

    # ---------------- Land
    "terrain": "terrain",
    "ferme": "terrain",

    # ---------------- Commercial
    "local commercial": "local commercial",
    "commercialpremise": "local commercial",
    "fond": "local commercial",
    "atelier": "local commercial",
    "surfaces": "local commercial",
    "entrepôt": "local commercial",

    # ---------------- Office
    "bureau": "bureau",
    "office": "bureau",

    # ---------------- large assets / other
    "immeuble": "autre",
    "autre": "autre",
    "gérance": "autre",
}

In [109]:
dataset_full_raw["type"] = (
    dataset_full_raw["type"]
    .astype("string")
    .str.strip()
    .str.lower()
)
dataset_full_raw["type"] = (
    dataset_full_raw["type"]
    .map(TYPE_MAPPING)
    .fillna("Autre")
)
dataset_full_raw["type"].unique()

array(['terrain', 'Autre', 'maison', 'appartement', 'autre', 'bureau',
       'local commercial'], dtype=object)

In [110]:
dataset_full_raw["contrat"].unique()

array(['Vente', 'Location', 'Bureaux & Commerces', 'Terrain',
       'Location vacances', 'Partage', 'Achat', 'location', nan, 'rental',
       'sale', 'shortTermRental'], dtype=object)

In [111]:
CONTRAT_MAPPING = {

    # -------- Sale
    "vente": "vente",
    "sale": "vente",
    
    "terrain": "vente",  # usually land sale listings

    # -------- Rental
    "location": "location",
    "Location": "location",
    "rental": "location",
}

In [112]:
dataset_full_raw["contrat"] = (
    dataset_full_raw["contrat"]
    .astype("string")
    .str.strip()
    .str.lower()
)
dataset_full_raw["contrat"] = (
    dataset_full_raw["contrat"]
    .map(CONTRAT_MAPPING)
    .fillna("Autre")
)

dataset_full_raw["contrat"].unique()

array(['vente', 'location', 'Autre'], dtype=object)

In [113]:
dataset_full_raw["contrat"].value_counts()

contrat
location    110991
vente        30343
Autre        10413
Name: count, dtype: int64

In [114]:
dataset_full_raw["pieces"].unique()

<FloatingArray>
[<NA>, 3.0, 6.0, 5.0, 4.0, 2.0, 1.0, 8.0, 7.0]
Length: 9, dtype: Float64

In [115]:
dataset_full_raw["etage"].unique()

<IntegerArray>
[<NA>, 2, 5, 3, 1, 0, 4, 6, 7, 9, -1, 8]
Length: 12, dtype: Int64

In [116]:
dataset_full_raw["standing"].unique()

array([None, 'Moyen standing', 'Haut standing', 'Populaire', 'Ancien',
       nan], dtype=object)

In [117]:
dataset_full_raw["adresse"].unique()  ## needs to be processed further 

array([None, 'Ariana', 'El Omrane Superieur', ...,
       'Cité Hedi Khlil sur Route Principale de Balta Bouaouane',
       'Clinique Rawabi', 'Bonne Affaires'], dtype=object)

In [118]:
dataset_full_raw["ville"].unique()

array([nan, "étage d'un immeuble à Tronja, Tunis",
       'situé à la Médina, Tunis', 'situé à Kef Abbed, Bizerte',
       'm situé à Borj Louzir, Ariana',
       'sur une route goudrounée à Menzel Bouzelfa, Nabeul',
       'étage situé dans une résidence à Centre Ville, Tunis',
       'éme étage situé dans une résidence avec ascenseur à Lafayette, Tunis',
       "er étage situé dans une résidence d'EL Passage, Tunis",
       'éme étage situé dans un immeuble à Centre ville, Tunis',
       'Fares El Khouri, Tunis',
       'avec une maison située à Menzel Bouzalfa, Nabeul',
       'situé sur la route principale à Menzel Bouzelfa, Nabeul',
       'Nichée Dans Un Quartier Résidentiel À Hammam Chatt, Ben Arous',
       'Située À Borj Louzir, Ariana',
       'Il Se Trouve À Cité El Hédi Nouira, Ariana',
       'DT Description Découvrez cetta magnifique VILLA spacieuse et lumineuse située à Ennkhilet, Ariana',
       'une résidence à Borj Louzir, Ariana',
       'au coeur du Centre Ville, Tu

In [119]:
dataset_full_raw["gouvernerat"].unique()

array([None, 'Ariana', 'Tunis', 'Nabeul', 'Sfax', 'Bizerte', 'Sousse',
       'Kairouan', 'Manouba', 'Ben arous', 'Zaghouan', 'Medenine',
       'Monastir', 'Jendouba', 'Beja', 'Gabes', 'Mahdia', 'Tataouine',
       'Sidi bouzid', 'Siliana', 'Le Kef', 'Kebili', 'Gafsa', 'Kasserine',
       'Tozeur', 'bizerte', 'cap-bon', 'grand-tunis', 'kairouan',
       'mahdia', 'monastir', 'sfax', 'sousse', nan, 'Sahloul', 'Hammamet',
       'Ben Arous', 'Séjoumi', 'Medina', 'Ezzahra', 'La Marsa', 'Mourouj',
       'Mégrine', 'Megrine', 'Kram', 'Gabès', 'Cap Bon', 'Kelibia',
       'Benzert', 'Médiouna', 'Benzerit', 'Oued Ghenim Jawhra', 'Djerba',
       'Sidi El Bechir', 'Sidi Bouzid', 'Bardo', 'Marrakech', 'Haouaria',
       'Oued Erroumin', 'Kéliblia', 'Mahres', 'Tunisie', 'Kef',
       'non spécifié', 'Enfidha', 'صفاقس', 'Béja', 'Kerkennah', 'SFAX',
       'GP5', 'Borj Zwara', 'Sanhaja', "M'ghira", 'Hammamet Nord',
       'Kibili', 'Gabés', 'Zarzis', 'قصرين'], dtype=object)

In [120]:
TUNISIA_GOUVERNORATS = [
    "tunis","ariana","ben arous","manouba",
    "nabeul","zaghouan","bizerte","beja","jendouba","le kef","siliana",
    "sousse","monastir","mahdia","sfax","kairouan","kasserine","sidi bouzid",
    "gabes","medenine","tataouine","gafsa","tozeur","kebili"
]
GOUVERNORAT_MAPPING = {

    # ---- Grand Tunis
    "tunis": "tunis",
    "ariana": "ariana",
    "ben arous": "ben arous",
    "manouba": "manouba",
    "grand-tunis": "tunis",

    # ---- North / Cap Bon
    "nabeul": "nabeul",
    "cap-bon": "nabeul",
    "bizerte": "bizerte",

    # ---- Northwest
    "beja": "beja",
    "jendouba": "jendouba",
    "le kef": "le kef",
    "siliana": "siliana",

    # ---- Sahel
    "sousse": "sousse",
    "monastir": "monastir",
    "mahdia": "mahdia",

    # ---- Center
    "sfax": "sfax",
    "kairouan": "kairouan",
    "kasserine": "kasserine",
    "sidi bouzid": "sidi bouzid",

    # ---- South
    "gabes": "gabes",
    "medenine": "medenine",
    "tataouine": "tataouine",
    "gafsa": "gafsa",
    "tozeur": "tozeur",
    "kebili": "kebili",
}

In [121]:
dataset_full_raw["gouvernerat"] = (
    dataset_full_raw["gouvernerat"]
    .astype("string")
    .str.strip()
    .str.lower()
    .map(GOUVERNORAT_MAPPING)
)

In [122]:
dataset_full_raw[["titre", "description", "caracteristiques"]] = (
    dataset_full_raw[["titre", "description", "caracteristiques"]]
    .astype("string")
)

In [123]:
dataset_full_raw.to_csv("clean_dataset.csv")


In [125]:
dataset_full_raw["contrat"].value_counts()

contrat
location    110991
vente        30343
Autre        10413
Name: count, dtype: int64